# B7 · Du notebook à la production — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b7_production/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b7_production/03_notebook.ipynb`, créée par `python tools/start_chapter.py B7`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B7.Q1 à B7.Q11, B7.R1 à B7.R3, B7.1 à B7.4 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B7.9 | Du notebook au paquet : data.py, model.py, train.py, pyproject.toml et installation éditable | 📦 | ★★★ | 60 |
|  | B7.10 | Une interface en ligne de commande : argparse, logging et codes de sortie | 📦 | ★★ | 30 |
|  | B7.11 | Configuration : dataclass + YAML, surcharges en ligne de commande, validation | 📦 | ★★ | 30 |
|  | B7.12 | Prédire : même graine, puis num_workers = 4, puis GPU — résultats identiques ? | 🔮 | ★ | 10 |
|  | B7.13 | Mesurer le non-déterminisme : graines fixées ou non, algorithmes déterministes, cinq exécutions | 🔬 | ★★ | 30 |
|  | B7.14 | Suivi d'expériences maison : identifiant de run, configuration, métriques et artefacts en JSONL | 📦 | ★★ | 30 |
|  | B7.15 | Tests pytest pour le ML : formes, overfitting d'un mini-batch, prétraitement, schéma des données | 📦 | ★★★ | 45 |
| B | B7.16 | Le modèle qui marche dans le notebook et pas en production : cinq écarts à retrouver | 🐛 | ★★ | 30 |
| B | B7.17 à B7.21 | le service (suite) : l'artefact, l'API, la latence, l'accélération sur CPU et l'export (*prochaine session de génération*) | | | |
| C | B7.22 à B7.25 | l'exploitation : la dérive, l'intégration continue, le Dockerfile et le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B7")  # your own library

## Objectifs et rappel express

- Restructurer un notebook en paquet Python installable, avec une ligne de commande, du logging et une configuration validée.
- Rendre un entraînement reproductible et traçable (graines, déterminisme, versions, empreinte des données, journal des runs), et rapporter un score sur plusieurs graines avec son intervalle.
- Écrire des tests utiles à un projet de ML et les automatiser dans une intégration continue.
- Empaqueter un modèle avec son prétraitement, le servir par une API HTTP et mesurer sa latence.
- Exporter et accélérer l'inférence sur CPU en vérifiant l'équivalence numérique ; détecter une dérive des données et décider d'une action.

**Rappel express.** Intervalle sur $n$ graines : $\bar a \pm t_{0{,}975;\,n-1}\,\frac{s}{\sqrt n}$ ($s$ divisé par $n - 1$). Percentile de `np.percentile` : position $h = \frac{q}{100}(n - 1)$ dans les mesures triées, interpolation linéaire. Débit $= \frac{\text{taille du batch}}{\text{latence du batch}}$ ; loi de Little $L = \lambda W$. Taille des poids $= N \times$ octets par paramètre (4, 2, 1). $\operatorname{PSI} = \sum_k (p_k - q_k)\ln\frac{p_k}{q_k} = D_{\text{KL}}(p\,\|\,q) + D_{\text{KL}}(q\,\|\,p)$ ; $D = \sup_x |F_{\text{réf}}(x) - F_{\text{prod}}(x)|$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B7.1 à B7.4)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres ensemble, entre guillemets, dans n'importe quel ordre (`"EF"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », l'estimation 🧮 B7.5, la réflexion (🗣️ B7.6, ⚖️ B7.7, 📄 B7.8) et l'entretien se corrigent avec `05_solutions.md`.

**B7.Q1 — Les pièges du notebook : état caché et ordre d'exécution**

In [ ]:
# B7.Q1: the values you found on paper
answer_B7_Q1a = ...  # a) a whole number
answer_B7_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q1c = ...  # c) True or False
answer_B7_Q1d = ...  # d) the letter of your choice, e.g. "E"
answer_B7_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B7_Q1a, answer_B7_Q1b, answer_B7_Q1c, answer_B7_Q1d, answer_B7_Q1e]):
    wb.check(f"B7.Q1{letter}", answer)

**B7.Q2 — src/, tests/, configs/, scripts/ : où va quoi ?**

In [ ]:
# B7.Q2: the values you found on paper
answer_B7_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q2b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q2c = ...  # c) the letter of your choice, e.g. "E"
answer_B7_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_B7_Q2e = ...  # e) True or False
answer_B7_Q2f = ...  # f) the letter of your choice, e.g. "E"
answer_B7_Q2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B7_Q2a, answer_B7_Q2b, answer_B7_Q2c, answer_B7_Q2d, answer_B7_Q2e, answer_B7_Q2f, answer_B7_Q2g]):
    wb.check(f"B7.Q2{letter}", answer)

**B7.Q3 — print ou logging : niveaux, destinations et messages utiles**

In [ ]:
# B7.Q3: the values you found on paper
answer_B7_Q3a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q3b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q3c = ...  # c) a whole number
answer_B7_Q3d = ...  # d) True or False
answer_B7_Q3e = ...  # e) the letter of your choice, e.g. "E"
answer_B7_Q3f = ...  # f) a whole number

for letter, answer in zip("abcdef", [answer_B7_Q3a, answer_B7_Q3b, answer_B7_Q3c, answer_B7_Q3d, answer_B7_Q3e, answer_B7_Q3f]):
    wb.check(f"B7.Q3{letter}", answer)

**B7.Q4 — Ce qui va dans la configuration, et ce qui ne doit jamais y aller**

In [ ]:
# B7.Q4: the values you found on paper
answer_B7_Q4a = ...  # a) the letters, e.g. "EF"
answer_B7_Q4b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q4c = ...  # c) a whole number
answer_B7_Q4d = ...  # d) True or False
answer_B7_Q4e = ...  # e) the letter of your choice, e.g. "E"
answer_B7_Q4f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B7_Q4a, answer_B7_Q4b, answer_B7_Q4c, answer_B7_Q4d, answer_B7_Q4e, answer_B7_Q4f]):
    wb.check(f"B7.Q4{letter}", answer)

**B7.Q5 — Même graine, résultats différents : les sources de non-déterminisme**

In [ ]:
# B7.Q5: the values you found on paper
answer_B7_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q5b = ...  # b) True or False
answer_B7_Q5c = ...  # c) the letter of your choice, e.g. "E"
answer_B7_Q5d = ...  # d) True or False
answer_B7_Q5e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B7_Q5a, answer_B7_Q5b, answer_B7_Q5c, answer_B7_Q5d, answer_B7_Q5e]):
    wb.check(f"B7.Q5{letter}", answer)

**B7.Q6 — requirements.txt, fichier de verrouillage, pyproject.toml : qui fige quoi ?**

In [ ]:
# B7.Q6: the values you found on paper
answer_B7_Q6a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q6b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q6c = ...  # c) True or False
answer_B7_Q6d = ...  # d) the letter of your choice, e.g. "E"
answer_B7_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B7_Q6a, answer_B7_Q6b, answer_B7_Q6c, answer_B7_Q6d, answer_B7_Q6e]):
    wb.check(f"B7.Q6{letter}", answer)

**B7.Q7 — Que tester dans un projet de machine learning ?**

In [ ]:
# B7.Q7: the values you found on paper
answer_B7_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q7b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_B7_Q7d = ...  # d) True or False
answer_B7_Q7e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B7_Q7a, answer_B7_Q7b, answer_B7_Q7c, answer_B7_Q7d, answer_B7_Q7e]):
    wb.check(f"B7.Q7{letter}", answer)

**B7.Q8 — Artefact de modèle : les poids seuls ne suffisent pas**

In [ ]:
# B7.Q8: the values you found on paper
answer_B7_Q8a = ...  # a) the letters, e.g. "EF"
answer_B7_Q8b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q8c = ...  # c) the letter of your choice, e.g. "E"
answer_B7_Q8d = ...  # d) True or False
answer_B7_Q8e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B7_Q8a, answer_B7_Q8b, answer_B7_Q8c, answer_B7_Q8d, answer_B7_Q8e]):
    wb.check(f"B7.Q8{letter}", answer)

**B7.Q9 — Latence p50, p95 et débit : ce que ressent l'utilisateur**

In [ ]:
# B7.Q9: the values you found on paper
answer_B7_Q9a = ...  # a) a whole number
answer_B7_Q9b = ...  # b) a whole number
answer_B7_Q9c = ...  # c) True or False
answer_B7_Q9d = ...  # d) a whole number
answer_B7_Q9e = ...  # e) a whole number
answer_B7_Q9f = ...  # f) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdef", [answer_B7_Q9a, answer_B7_Q9b, answer_B7_Q9c, answer_B7_Q9d, answer_B7_Q9e, answer_B7_Q9f]):
    wb.check(f"B7.Q9{letter}", answer)

**B7.Q10 — torch.export, ONNX, TorchScript : lequel choisir aujourd'hui ?**

In [ ]:
# B7.Q10: the values you found on paper
answer_B7_Q10a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q10c = ...  # c) True or False
answer_B7_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_B7_Q10e = ...  # e) True or False
answer_B7_Q10f = ...  # f) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdef", [answer_B7_Q10a, answer_B7_Q10b, answer_B7_Q10c, answer_B7_Q10d, answer_B7_Q10e, answer_B7_Q10f]):
    wb.check(f"B7.Q10{letter}", answer)

**B7.Q11 — Écart entraînement-production et dérive des données : vrai ou faux**

In [ ]:
# B7.Q11: the values you found on paper
answer_B7_Q11a = ...  # a) True or False
answer_B7_Q11b = ...  # b) True or False
answer_B7_Q11c = ...  # c) True or False
answer_B7_Q11d = ...  # d) True or False
answer_B7_Q11e = ...  # e) the letter of your choice, e.g. "E"
answer_B7_Q11f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B7_Q11a, answer_B7_Q11b, answer_B7_Q11c, answer_B7_Q11d, answer_B7_Q11e, answer_B7_Q11f]):
    wb.check(f"B7.Q11{letter}", answer)

**B7.R1 — Ce qu'une ligne de journal d'expériences doit contenir**

In [ ]:
# B7.R1: the values you found on paper
answer_B7_R1a = ...  # a) the letters, e.g. "EF"
answer_B7_R1b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_R1c = ...  # c) True or False
answer_B7_R1d = ...  # d) a whole number
answer_B7_R1e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B7_R1a, answer_B7_R1b, answer_B7_R1c, answer_B7_R1d, answer_B7_R1e]):
    wb.check(f"B7.R1{letter}", answer)

**B7.R2 — Checkpoint complet, state_dict et weights_only**

In [ ]:
# B7.R2: the values you found on paper
answer_B7_R2a = ...  # a) the letters, e.g. "EF"
answer_B7_R2b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_R2c = ...  # c) True or False
answer_B7_R2d = ...  # d) True or False
answer_B7_R2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B7_R2a, answer_B7_R2b, answer_B7_R2c, answer_B7_R2d, answer_B7_R2e]):
    wb.check(f"B7.R2{letter}", answer)

**B7.R3 — Sauvegarder un Pipeline et figer les versions**

In [ ]:
# B7.R3: the values you found on paper
answer_B7_R3a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_R3b = ...  # b) True or False
answer_B7_R3c = ...  # c) the letter of your choice, e.g. "E"
answer_B7_R3d = ...  # d) True or False
answer_B7_R3e = ...  # e) the letters, e.g. "EF"
answer_B7_R3f = ...  # f) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdef", [answer_B7_R3a, answer_B7_R3b, answer_B7_R3c, answer_B7_R3d, answer_B7_R3e, answer_B7_R3f]):
    wb.check(f"B7.R3{letter}", answer)

**Ex B7.1 — Taille d'un artefact : float32, bfloat16, int8, et la mémoire au chargement**

In [ ]:
# Ex B7.1: the values you found on paper
answer_B7_1a = ...  # a) a number, 1 decimal
answer_B7_1b = ...  # b) a number, 1 decimal
answer_B7_1c = ...  # c) a number, 1 decimal
answer_B7_1d = ...  # d) a number, 1 decimal
answer_B7_1e = ...  # e) a number, 1 decimal
answer_B7_1f = ...  # f) a number, 1 decimal
answer_B7_1g = ...  # g) a number, 1 decimal
answer_B7_1h = ...  # h) a whole number
answer_B7_1i = ...  # i) a whole number
answer_B7_1j = ...  # j) a number, 1 decimal

for letter, answer in zip("abcdefghij", [answer_B7_1a, answer_B7_1b, answer_B7_1c, answer_B7_1d, answer_B7_1e, answer_B7_1f, answer_B7_1g, answer_B7_1h, answer_B7_1i, answer_B7_1j]):
    wb.check(f"B7.1{letter}", answer)

**Ex B7.2 — Cinq graines, un intervalle : la dispersion d'une accuracy rapportée**

In [ ]:
# Ex B7.2: the values you found on paper
answer_B7_2a = ...  # a) a number, 1 decimal
answer_B7_2b = ...  # b) a number, 3 decimals
answer_B7_2c = ...  # c) a number, 3 decimals
answer_B7_2d = ...  # d) a number, 2 decimals
answer_B7_2e = ...  # e) a number, 2 decimals
answer_B7_2f = ...  # f) True or False
answer_B7_2g = ...  # g) a number, 1 decimal
answer_B7_2h = ...  # h) a number, 2 decimals
answer_B7_2i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_B7_2a, answer_B7_2b, answer_B7_2c, answer_B7_2d, answer_B7_2e, answer_B7_2f, answer_B7_2g, answer_B7_2h, answer_B7_2i]):
    wb.check(f"B7.2{letter}", answer)

**Ex B7.3 — Latence et débit : p50 et p95 sur vingt mesures, batch 1 contre batch 32**

In [ ]:
# Ex B7.3: the values you found on paper
answer_B7_3a = ...  # a) a number, 2 decimals
answer_B7_3b = ...  # b) a number, 2 decimals
answer_B7_3c = ...  # c) a number, 2 decimals
answer_B7_3d = ...  # d) a whole number
answer_B7_3e = ...  # e) a whole number
answer_B7_3f = ...  # f) a whole number
answer_B7_3g = ...  # g) a number, 1 decimal
answer_B7_3h = ...  # h) a number, 2 decimals
answer_B7_3i = ...  # i) True or False
answer_B7_3j = ...  # j) a number, 2 decimals

for letter, answer in zip("abcdefghij", [answer_B7_3a, answer_B7_3b, answer_B7_3c, answer_B7_3d, answer_B7_3e, answer_B7_3f, answer_B7_3g, answer_B7_3h, answer_B7_3i, answer_B7_3j]):
    wb.check(f"B7.3{letter}", answer)

**Ex B7.4 — Indice de stabilité de population (PSI) à la main sur quatre intervalles**

In [ ]:
# Ex B7.4: the values you found on paper
answer_B7_4a = ...  # a) a number, 4 decimals
answer_B7_4b = ...  # b) a number, 3 decimals
answer_B7_4c = ...  # c) a number, 3 decimals
answer_B7_4d = ...  # d) the letter of your choice, e.g. "E"
answer_B7_4e = ...  # e) a number, 3 decimals
answer_B7_4f = ...  # f) the letter of your choice, e.g. "E"
answer_B7_4g = ...  # g) True or False
answer_B7_4h = ...  # h) a number, 2 decimals
answer_B7_4i = ...  # i) a number, 3 decimals

for letter, answer in zip("abcdefghi", [answer_B7_4a, answer_B7_4b, answer_B7_4c, answer_B7_4d, answer_B7_4e, answer_B7_4f, answer_B7_4g, answer_B7_4h, answer_B7_4i]):
    wb.check(f"B7.4{letter}", answer)

## Partie A · Le projet : du notebook au paquet, ligne de commande, configuration, reproductibilité, suivi et tests

Au MP5, `cifarkit` rangeait déjà ton code dans un dossier que le notebook importait. Ici, ce code devient un **projet** : un paquet installable (B7.9), piloté par une ligne de commande (B7.10) et une configuration validée (B7.11), dont tu mesures la reproductibilité (B7.12, B7.13), qui journalise ses runs (B7.14) et que des tests protègent (B7.15). Le modèle est volontairement petit (le CNN de 207.6, 2 000 images, une epoch : quelques secondes) : on juge ici la mécanique du projet, pas l'accuracy.

La cellule suivante crée ton dossier de projet, `PROJECT` (`mon_travail/b7_production/cifar-prod/` ; le notebook de solutions travaille dans un dossier temporaire), y écrit `data/cifar10_2000.npz` (200 images par classe pour l'entraînement, 50 pour la validation, un sous-ensemble figé de CIFAR-10, `uint8` `(N, 32, 32, 3)`, avec les tableaux `x_train`, `y_train`, `x_val`, `y_val`) et charge `wbp` (`wb.prod`, l'infrastructure du chapitre : écrire un fichier, lancer une commande, installer dans un environnement neuf). Toutes les commandes partent du dossier du projet, avec des chemins absolus.

In [ ]:
# Tools of part A: the folder of your project, the data of the chapter, the commands; nothing here is your package
import contextlib
import io
import json
import os
import re
import sys
import tempfile
import time
import warnings
from pathlib import Path

import numpy as np
import torch
from wb import prod as wbp
from wb.nbtools import StopCheck, checked, fr, refuses, run_pytest_group, same, verdict, written

PROJECT = wbp.project_dir("learner")             # the folder of your project
DATA_FILE = wbp.write_subset(PROJECT / "data")  # the 2 000 + 500 images of the chapter (.npz, 7.7 MB, not for git)
SUBSET = wbp.cifar_subset()                     # the same arrays, for the checks
CLASSES = wbp.CLASSES
GITIGNORE = "data/\nruns/\n__pycache__/\n*.egg-info/\n.pytest_cache/\n"
DEFAULTS = {"seed": 0, "epochs": 1, "lr": 0.05, "batch_size": 128, "n_train": 2000, "arch": "cnn", "widths": [16, 32],
            "data": "data/cifar10_2000.npz", "deterministic": True, "num_workers": 0}   # TrainConfig().to_dict()


def scratch(prefix):
    """A new temporary folder (outside the project): the checks write their test files there."""
    return Path(tempfile.mkdtemp(prefix=f"wb_b7_{prefix}_"))


def run_line(ex_id, result, ok, success, failure):
    """One verdict about a command of your package: what was run, then the end of stderr when it went wrong (⏳ when
    the command stopped on a NotImplementedError of your skeleton)."""
    shown = result.command().replace(f"python -m {wbp.PACKAGE}", "cifarprod")
    pending = re.findall(r"NotImplementedError: (.+)", result.stderr)
    if not ok and pending:
        print(f"⏳ Ex {ex_id} : {shown} → pas encore fait ({pending[-1].strip()}).")
        return False
    if result.timed_out:
        ok, failure = False, f"délai dépassé ({result.seconds:.0f} s) : {failure}"
    seen = (f"stderr : {result.tail(2)[:400]}" if result.stderr.strip() else
            f"rien sur stderr ; stdout : {result.stdout.strip()[-300:]!r}" if result.stdout.strip() else "rien sur stderr")
    verdict(ex_id, ok, f"{shown} → {success}", f"{shown} → {failure} (code {result.returncode} ; {seen})")
    return ok


print(wbp.project_line(PROJECT, "learner"))
print(f"ℹ️ données : data/{DATA_FILE.name} ({len(SUBSET['x_train'])} images d'entraînement, {len(SUBSET['x_val'])} de "
      f"validation, 200 et 50 par classe ; empreinte {wbp.SUBSET_FINGERPRINT[:12]}…)")

### Ex B7.9 — Du notebook au paquet : data.py, model.py, train.py, pyproject.toml et installation éditable 📦 ★★★ ⏱️ 60 min
**Objectif :** transformer le code d'un notebook en paquet Python installable, avec un layout `src/`.  
**Prérequis :** MP5 (`cifarkit`) · ch. 23 · fiche §207.1 (« Une structure de dépôt », « pyproject.toml et installation éditable »), §207.4 (`seed_everything`, `array_fingerprint`), §207.6 · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

> **Mode d'emploi du paquet** : chaque fichier de ton paquet est une chaîne de la cellule à compléter (`DATA_PY = r'''...'''`) ; la cellule de vérification l'**écrit** dans ton projet (`mon_travail/b7_production/cifar-prod/`, à côté de ce notebook), puis l'importe ou le lance dans un autre processus. Pour corriger un fichier, corrige sa chaîne et réexécute les deux cellules. Si tu préfères un éditeur, modifie le fichier sur le disque : la cellule le voit et ne l'écrase plus (elle le dit).

**Le contrat.** Ton projet doit avoir cette forme (le `.gitignore` et le fichier de données sont fournis) :

```text
cifar-prod/
├── pyproject.toml        .gitignore        data/cifar10_2000.npz
└── src/cifarprod/
    ├── __init__.py       (__version__ = "0.1.0", fourni)
    ├── data.py           model.py           train.py
```

- `pyproject.toml` : le backend de construction de la fiche (`setuptools`, déjà installé : un autre, comme `hatchling`, n'est pas dans l'environnement du workbook) ; le paquet `cifarprod`, version `0.1.0`, Python ≥ 3.11 ; les dépendances avec des bornes basses (`numpy`, `torch`, `pyyaml`, `safetensors`) ; deux extras, `dev` (`pytest`) et `serve` (`fastapi`, `uvicorn`, `pydantic`) ; le point d'entrée `cifarprod = "cifarprod.cli:main"` (le fichier `cli.py` viendra en B7.10) ; et, pour B7.15, une table `[tool.pytest.ini_options]` avec `testpaths = ["tests"]` et le marqueur `slow` déclaré dans `markers`.
- `data.py` : `CLASSES` (les 10 noms, dans l'ordre des sorties) ; `load_npz(path, part="train")` rend `(X, y)`, `X` `uint8` `(N, 32, 32, 3)` et `y` `int64` ; elle **vérifie le schéma** : `ValueError` pour des images qui ne sont pas `uint8` `(N, 32, 32, 3)`, des labels qui ne sont pas $N$ entiers de 0 à 9, ou une autre partie que `"train"` et `"val"` ; `FileNotFoundError` pour un fichier absent ; `fingerprint(*arrays)`, l'empreinte de 207.4 (le type, la forme, puis les octets de chaque tableau) ; `channel_stats(images)` rend `(mean, std)`, trois floats Python chacun, la moyenne et l'écart-type (divisé par $n$) de chaque canal dans $[0, 1]$ ; `preprocess(images, mean, std)` rend un tenseur `float32` `(N, 3, 32, 32)`, chaque canal valant $(\text{pixel}/255 - \text{mean}_c)/\text{std}_c$, accepte aussi **une** image `(32, 32, 3)` (rendue `(1, 3, 32, 32)`) et refuse un tableau qui n'est pas `uint8` (`ValueError`) ; `make_loader(X, y, mean, std, batch_size=128, shuffle=False, seed=0, num_workers=0)` rend un `DataLoader` de mini-batches `(preprocess(X), y)`, `float32` et `int64`, dont l'ordre mélangé vient d'un `torch.Generator().manual_seed(seed)` **à lui** (il ne dépend pas du générateur global) ; `synthetic(n, seed=0)` rend $n$ images `uint8` tirées par `np.random.default_rng(seed).integers(0, 256, (n, 32, 32, 3), dtype=np.uint8)` et les labels `np.arange(n) % 10` (pour les tests et l'intégration continue, qui n'ont pas CIFAR-10).
- `model.py` : `build_model(arch="cnn", widths=(12, 24), n_classes=10)` rend un `nn.Sequential` des couches de la docstring, **dans cet ordre** (`"cnn"`, le CNN de la fiche et de `cifarkit.small_cnn` ; `"vgg"`, celui de `cifarkit.vgg` avec batchnorm et max pooling) : les noms de ses poids (`0.weight`, `1.weight`…) sont alors ceux de `cifarkit`, et le modèle livré du MP5 s'y recharge ; `ValueError` pour une autre architecture ou des largeurs vides ; `n_parameters(model)`.
- `train.py` : `seed_everything(seed, deterministic=True)` (207.4) ; `evaluate(model, loader)` rend `(loss moyenne, accuracy)` en mode `eval()`, sans graphe, puis rend au modèle son mode d'avant ; `train(X, y, X_val, y_val, *, arch="cnn", widths=(16, 32), epochs=1, lr=0.05, batch_size=128, seed=0, deterministic=True, num_workers=0)` appelle d'abord `seed_everything(seed, deterministic)`, apprend la normalisation sur `X` **seulement**, entraîne avec SGD (`momentum=0.9`) et la cross-entropy sur les mini-batches de `make_loader(..., shuffle=True, seed=seed)`, et rend `{"model": le modèle en mode eval(), "mean", "std", "history": {"train_loss", "val_loss", "val_acc"} (un float par epoch), "seconds"}`. **Aucun `print`** : une ligne INFO par epoch et une ligne DEBUG par mini-batch avec `log = logging.getLogger(__name__)` (207.2), que B7.10 affichera.

La cellule de vérification écrit tes fichiers, importe ton paquet **neuf** depuis `src/` et compare chaque fonction : `load_npz` aux tableaux du fichier et quatre fichiers mal formés, `fingerprint` à `hashlib`, `channel_stats` et `preprocess` à NumPy, `make_loader` avec et sans mélange (le générateur global changé entre deux appels), `build_model` à l'architecture du contrat (mêmes tenseurs et mêmes formes, quatre réglages), puis lance `train` deux fois avec la même graine (mêmes poids) et une fois avec une autre, et l'entraînement fumée des réglages par défaut sur les 2 000 images (moins d'une minute, une accuracy de validation au-dessus du hasard). Enfin, elle fait `pip install -e .` de ton projet dans un **environnement virtuel neuf** (qui voit les bibliothèques du notebook, mais pas ton dossier `src/`) et vérifie, depuis un autre dossier, que `import cifarprod` lit ton dossier `src/` (une installation éditable, pas une copie), la version et le point d'entrée. Ton environnement à toi n'est jamais modifié : pour avoir la commande `cifarprod` dans un terminal, lance toi-même `pip install -e .` dans le dossier du projet.

Dans tes notes : pourquoi le layout `src/` empêche-t-il `import cifarprod` depuis la racine du projet tant que le paquet n'est pas installé, et qu'est-ce que cela protège ? Pourquoi `train` apprend-il la normalisation sur `X` seulement, et que deviendrait l'évaluation si on la calculait sur l'entraînement et la validation ensemble (ch. 12) ? Combien de secondes prend l'entraînement fumée, et que juge-t-il (et que ne juge-t-il pas) ?

In [ ]:
# Ex B7.9: the files of your package, as strings. The check cell writes them into PROJECT, then imports and runs
# them; to change a file, change its string here and run both cells again.
PYPROJECT_TOML = ...   # the pyproject.toml of the contract (start from the one of the fiche, 207.1)

# src/cifarprod/__init__.py (given)
INIT_PY = r'''"""cifarprod: a CIFAR-10 classifier as an installable package (bonus chapter B7)."""

__version__ = "0.1.0"
'''

# src/cifarprod/data.py
DATA_PY = r'''"""Data of cifarprod: the CIFAR-10 subset file, its schema, its fingerprint, the normalization and the loaders."""

from __future__ import annotations

import hashlib
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

CLASSES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")


def load_npz(path, part="train"):
    """(X, y) of one part ("train" or "val") of a .npz file with the arrays x_train, y_train, x_val, y_val: X uint8
    (N, 32, 32, 3), y int64 (N,). The schema is checked: ValueError for images that are not uint8 (N, 32, 32, 3), for
    labels that are not N integers in 0..9, or for another part; FileNotFoundError if the file does not exist."""
    raise NotImplementedError("load_npz")


def fingerprint(*arrays):
    """SHA-256 (64 hexadecimal characters) of the dtype, the shape and the bytes of each array, in this order (207.4)."""
    raise NotImplementedError("fingerprint")


def channel_stats(images):
    """(mean, std): three Python floats each, the mean and the standard deviation (divided by n) of the pixels of each
    channel on the scale [0, 1], over all the uint8 images (N, H, W, 3) given."""
    raise NotImplementedError("channel_stats")


def preprocess(images, mean, std):
    """uint8 RGB images (N, 32, 32, 3), or one image (32, 32, 3), as a float32 tensor (N, 3, 32, 32) (N = 1 for one
    image), each channel c equal to (pixel / 255 - mean[c]) / std[c]. ValueError for an array that is not uint8."""
    raise NotImplementedError("preprocess")


def make_loader(X, y, mean, std, batch_size=128, shuffle=False, seed=0, num_workers=0):
    """A DataLoader of (preprocess(X), y) mini-batches, float32 (B, 3, 32, 32) and int64 (B,); when shuffle, the order
    comes from its own torch.Generator().manual_seed(seed)."""
    raise NotImplementedError("make_loader")


def synthetic(n, seed=0):
    """(X, y): n random uint8 images (n, 32, 32, 3) drawn by np.random.default_rng(seed), and the labels i % 10 (for the
    tests and the continuous integration, which have no CIFAR-10)."""
    raise NotImplementedError("synthetic")
'''

# src/cifarprod/model.py
MODEL_PY = r'''"""The models of cifarprod: "cnn" and "vgg", the CNNs of the MP5, rebuilt from a few settings."""

from __future__ import annotations

import torch.nn as nn


def build_model(arch="cnn", widths=(12, 24), n_classes=10):
    """The network `arch` with one stage per width, from float32 (N, 3, 32, 32) to the logits (N, n_classes).
    - "cnn": for each width w, Conv2d(c, w, 3, padding=1), ReLU, MaxPool2d(2); then Flatten and Linear;
    - "vgg": for each width w, twice Conv2d(c, w, 3, padding=1, bias=False), BatchNorm2d(w), ReLU; a MaxPool2d(2)
      after every stage but the last; then AdaptiveAvgPool2d(1), Flatten, Linear(widths[-1], n_classes).
    ValueError for another arch or for empty widths."""
    raise NotImplementedError("build_model")


def n_parameters(model):
    """The number of trainable parameters of a model."""
    raise NotImplementedError("n_parameters")
'''

# src/cifarprod/train.py
TRAIN_PY = r'''"""The training of cifarprod: the seeds, one evaluation, and train(), which returns the model with its normalization."""

from __future__ import annotations

import logging
import random
import time

import numpy as np
import torch
import torch.nn.functional as F

from .data import channel_stats, make_loader
from .model import build_model

log = logging.getLogger(__name__)


def seed_everything(seed, deterministic=True):
    """Seed Python, NumPy and PyTorch, switch the deterministic algorithms on or off, and return a generator seeded
    with `seed` for the DataLoader (207.4)."""
    raise NotImplementedError("seed_everything")


def evaluate(model, loader):
    """(mean cross-entropy, accuracy) of the model on all the examples of the loader, in eval() mode and without a
    graph; the mode of the model is restored afterwards."""
    raise NotImplementedError("evaluate")


def train(X, y, X_val, y_val, *, arch="cnn", widths=(16, 32), epochs=1, lr=0.05, batch_size=128, seed=0,
          deterministic=True, num_workers=0):
    """Train build_model(arch, widths) on the uint8 images X (N, 32, 32, 3) and their labels y: seed_everything(seed,
    deterministic) first, the normalization learned on X only, SGD(lr, momentum=0.9), the cross-entropy, the
    mini-batches of make_loader(..., shuffle=True, seed=seed). No print: one INFO line per epoch and one DEBUG line
    per mini-batch through `log`. Returns {"model": the trained model in eval() mode, "mean", "std", "history":
    {"train_loss", "val_loss", "val_acc"} (one float per epoch), "seconds"}."""
    raise NotImplementedError("train")
'''

In [ ]:
FILES_9 = {"pyproject.toml": PYPROJECT_TOML, ".gitignore": GITIGNORE, "src/cifarprod/__init__.py": INIT_PY,
           "src/cifarprod/data.py": DATA_PY, "src/cifarprod/model.py": MODEL_PY, "src/cifarprod/train.py": TRAIN_PY}
ok_9 = {}
with checked("B7.9"):
    wbp.write_files(PROJECT, FILES_9)
    missing_9 = [rel for rel, text in FILES_9.items() if text is ...]
    print("ℹ️ fichiers écrits dans le projet : " + ", ".join(r for r, t in FILES_9.items() if t is not ...)
          + (f" ; pas encore écrits : {', '.join(missing_9)}" if missing_9 else ""))
    pkg_9 = wbp.import_package(PROJECT)
    ok_9["import"] = getattr(pkg_9, "__version__", None) == "0.1.0"
    verdict("B7.9", ok_9["import"], "import cifarprod depuis src/ : version 0.1.0.",
            f"cifarprod.__version__ vaut {getattr(pkg_9, '__version__', None)!r}, attendu '0.1.0'.")

with checked("B7.9"):                                    # data.py
    data_9 = wbp.import_module(PROJECT, "data")
    X_9, y_9 = data_9.load_npz(DATA_FILE, "train")
    Xv_9, yv_9 = data_9.load_npz(DATA_FILE, "val")
    got_9 = [(np.asarray(a).dtype, np.asarray(a).shape) for a in (X_9, y_9, Xv_9, yv_9)]
    ok_9["load"] = (isinstance(X_9, np.ndarray) and X_9.dtype == np.uint8 and np.asarray(y_9).dtype == np.int64
                    and np.array_equal(X_9, SUBSET["x_train"]) and np.array_equal(y_9, SUBSET["y_train"])
                    and np.array_equal(Xv_9, SUBSET["x_val"]) and np.array_equal(yv_9, SUBSET["y_val"]))
    verdict("B7.9", ok_9["load"], "load_npz : les 2 000 images d'entraînement et les 500 de validation, uint8 et int64.",
            f"load_npz ne rend pas les tableaux du fichier en uint8 et int64 : reçu (dtype, forme) {got_9}.")
    bad_9 = scratch("bad")
    zeros_9 = np.zeros((4, 32, 32, 3), np.uint8)
    for name_9, (xb_9, yb_9) in {"des images float32": (zeros_9.astype(np.float32), np.arange(4)),
                                 "des images 32 × 32 × 1": (zeros_9[..., :1], np.arange(4)),
                                 "un label 10": (zeros_9, np.array([0, 1, 2, 10])),
                                 "3 labels pour 4 images": (zeros_9, np.arange(3))}.items():
        np.savez(bad_9 / "bad.npz", x_train=xb_9, y_train=yb_9, x_val=xb_9, y_val=yb_9)
        ok_9[name_9] = refuses("B7.9", f"load_npz d'un fichier avec {name_9}",
                               lambda: data_9.load_npz(bad_9 / "bad.npz", "train"))
    ok_9["part"] = refuses("B7.9", 'load_npz(..., "test")', lambda: data_9.load_npz(DATA_FILE, "test"))
    ok_9["absent"] = refuses("B7.9", "load_npz d'un fichier absent", lambda: data_9.load_npz(bad_9 / "absent.npz"),
                             error=FileNotFoundError)
    fp_9 = data_9.fingerprint(SUBSET["x_train"][:5], SUBSET["y_train"][:5])
    ok_9["fingerprint"] = fp_9 == wbp.array_fingerprint(SUBSET["x_train"][:5], SUBSET["y_train"][:5])
    verdict("B7.9", ok_9["fingerprint"], f"fingerprint de 5 images et de leurs labels : {str(fp_9)[:12]}…, comme "
            "hashlib sur le type, la forme et les octets.", f"fingerprint de 5 images et de leurs labels : "
            f"{str(fp_9)[:16]!r}…, attendu un autre SHA-256 (le type, la forme, puis les octets de chaque tableau).")
    mean_9, std_9 = data_9.channel_stats(SUBSET["x_train"])
    want_9 = (SUBSET["x_train"] / 255.0).mean(axis=(0, 1, 2)), (SUBSET["x_train"] / 255.0).std(axis=(0, 1, 2))
    ok_9["stats"] = (len(mean_9) == len(std_9) == 3 and all(type(v) is float for v in [*mean_9, *std_9])
                     and np.allclose(mean_9, want_9[0], atol=1e-10) and np.allclose(std_9, want_9[1], atol=1e-10))
    verdict("B7.9", ok_9["stats"], f"channel_stats : moyennes {np.round(mean_9, 4).tolist()}, écarts-types "
            f"{np.round(std_9, 4).tolist()} (trois floats Python chacun).",
            f"channel_stats rend {mean_9!r}, {std_9!r} : attendu trois floats Python chacun, moyenne et écart-type "
            "(divisé par n) des pixels de chaque canal dans [0, 1].")
    four_9 = SUBSET["x_val"][:4]
    want_x_9 = ((four_9 / 255.0 - np.array(want_9[0])) / np.array(want_9[1])).transpose(0, 3, 1, 2)
    x_9 = data_9.preprocess(four_9.copy(), tuple(want_9[0]), tuple(want_9[1]))
    ok_9["preprocess"] = (isinstance(x_9, torch.Tensor) and x_9.dtype == torch.float32
                          and same("B7.9", "preprocess de 4 images de validation", x_9, want_x_9, rtol=0, atol=1e-5))
    if not (isinstance(x_9, torch.Tensor) and x_9.dtype == torch.float32):
        verdict("B7.9", False, "", f"preprocess doit rendre un tenseur float32, reçu {type(x_9).__name__} "
                f"{getattr(x_9, 'dtype', '')}.")
    one_9 = data_9.preprocess(four_9[0], tuple(want_9[0]), tuple(want_9[1]))
    ok_9["one"] = tuple(getattr(one_9, "shape", ())) == (1, 3, 32, 32)
    verdict("B7.9", ok_9["one"], "preprocess d'une seule image (32, 32, 3) : (1, 3, 32, 32).",
            f"preprocess d'une seule image (32, 32, 3) : attendu (1, 3, 32, 32), reçu {tuple(getattr(one_9, 'shape', ()))}.")
    ok_9["float"] = refuses("B7.9", "preprocess d'images en float", lambda: data_9.preprocess(four_9 / 255.0, *want_9))

    def order_9(seed, global_seed):
        torch.manual_seed(global_seed)                  # the order must not depend on the global generator
        loader = data_9.make_loader(SUBSET["x_val"][:64], np.arange(64), *want_9, batch_size=16, shuffle=True, seed=seed)
        return np.concatenate([np.asarray(t) for _, t in loader])

    with wbp.torch_state_kept():
        plain_9 = data_9.make_loader(SUBSET["x_val"][:64], SUBSET["y_val"][:64], *want_9, batch_size=16)
        xb_9, tb_9 = next(iter(plain_9))
        a_9, b_9, c_9 = order_9(0, 1), order_9(0, 2), order_9(1, 1)
    ok_9["loader"] = (tuple(xb_9.shape) == (16, 3, 32, 32) and xb_9.dtype == torch.float32 and tb_9.dtype == torch.int64
                      and np.allclose(xb_9.numpy(), data_9.preprocess(SUBSET["x_val"][:16], *want_9).numpy())
                      and np.array_equal(tb_9.numpy(), SUBSET["y_val"][:16]))
    verdict("B7.9", ok_9["loader"], "make_loader sans mélange : les 16 premières images prétraitées et leurs labels, "
            "float32 et int64.", f"make_loader sans mélange : attendu le batch (16, 3, 32, 32) float32 des 16 premières "
            f"images prétraitées et leurs labels int64, reçu {tuple(xb_9.shape)} {xb_9.dtype}, {tb_9.dtype}.")
    ok_9["order"] = sorted(a_9) == list(range(64)) and np.array_equal(a_9, b_9) and not np.array_equal(a_9, c_9)
    verdict("B7.9", ok_9["order"], "make_loader mélangé : le même ordre pour la même graine, quel que soit le "
            "générateur global ; un autre pour une autre graine.",
            "make_loader mélangé : attendu le même ordre des 64 images pour seed=0 après torch.manual_seed(1) ou (2), "
            "et un autre pour seed=1 ; " + ("l'ordre suit le générateur global." if not np.array_equal(a_9, b_9)
                                           else "seed ne change pas l'ordre." if np.array_equal(a_9, c_9)
                                           else "des images manquent ou sont répétées."))
    Xs_9, ys_9 = data_9.synthetic(12, seed=3)
    ok_9["synthetic"] = (np.asarray(Xs_9).dtype == np.uint8 and np.asarray(Xs_9).shape == (12, 32, 32, 3)
                         and np.array_equal(Xs_9, np.random.default_rng(3).integers(0, 256, (12, 32, 32, 3), dtype=np.uint8))
                         and np.array_equal(ys_9, np.arange(12) % 10))
    verdict("B7.9", ok_9["synthetic"], "synthetic(12, seed=3) : 12 images uint8 de default_rng(3) et les labels i % 10.",
            "synthetic(12, seed=3) : attendu np.random.default_rng(3).integers(0, 256, (12, 32, 32, 3), "
            "dtype=np.uint8) et les labels np.arange(12) % 10.")

with checked("B7.9"):                                    # model.py
    model_9 = wbp.import_module(PROJECT, "model")
    for arch_9, widths_9 in (("cnn", (12, 24)), ("cnn", (8, 16, 32)), ("vgg", (16, 32, 64)), ("vgg", (8,))):
        mine_9 = model_9.build_model(arch_9, widths_9)
        want_m_9 = wbp.build_reference(arch_9, widths_9)
        shapes_9 = {k: tuple(v.shape) for k, v in mine_9.state_dict().items()}
        ref_shapes_9 = {k: tuple(v.shape) for k, v in want_m_9.state_dict().items()}
        out_9 = mine_9.eval()(torch.zeros(5, 3, 32, 32))
        ok_9[arch_9 + str(widths_9)] = shapes_9 == ref_shapes_9 and tuple(out_9.shape) == (5, 10)
        extra_9 = sorted(set(shapes_9) - set(ref_shapes_9))
        lacking_9 = sorted(set(ref_shapes_9) - set(shapes_9))
        other_9 = sorted(k for k in set(shapes_9) & set(ref_shapes_9) if shapes_9[k] != ref_shapes_9[k])
        diff_9 = "; ".join(part for part in [f"en trop : {extra_9[:4]}" if extra_9 else "",
                                             f"manquants : {lacking_9[:4]}" if lacking_9 else "",
                                             f"d'autres formes : {other_9[:4]}" if other_9 else ""] if part)
        verdict("B7.9", ok_9[arch_9 + str(widths_9)],
                f"build_model({arch_9!r}, {widths_9}) : {model_9.n_parameters(mine_9):_} paramètres, les mêmes couches "
                "que l'architecture du contrat, sortie (5, 10).".replace("_", " "),
                f"build_model({arch_9!r}, {widths_9}) : les noms et les formes des tenseurs du state_dict diffèrent de "
                f"ceux de l'architecture du contrat ({diff_9 or 'mêmes tenseurs'}) ; sortie {tuple(out_9.shape)}, "
                "attendu (5, 10).")
    n_9 = model_9.n_parameters(wbp.build_reference("cnn", (12, 24)))
    ok_9["n_parameters"] = n_9 == sum(p.numel() for p in wbp.build_reference("cnn", (12, 24)).parameters())
    verdict("B7.9", ok_9["n_parameters"], "n_parameters compte les paramètres entraînables.",
            f"n_parameters du CNN (12, 24) de la fiche : reçu {n_9}, attendu le nombre de valeurs de ses paramètres.")
    ok_9["default"] = ({k: tuple(v.shape) for k, v in model_9.build_model().state_dict().items()}
                       == {k: tuple(v.shape) for k, v in wbp.build_reference("cnn", (12, 24)).state_dict().items()})
    verdict("B7.9", ok_9["default"], "build_model() : le CNN (12, 24) de la fiche.",
            "build_model() sans argument doit construire le CNN (12, 24) de la fiche (arch=\"cnn\", widths=(12, 24)).")
    ok_9["arch"] = refuses("B7.9", 'build_model("resnet")', lambda: model_9.build_model("resnet"))

TRAINED_9 = None
with checked("B7.9"):                                    # train.py, in this notebook
    train_9 = wbp.import_module(PROJECT, "train")
    small_9 = dict(arch="cnn", widths=(8, 16), epochs=2, lr=0.05, batch_size=64, seed=0)
    printed_9 = io.StringIO()
    with wbp.torch_state_kept(), contextlib.redirect_stdout(printed_9):
        r1_9 = train_9.train(SUBSET["x_train"][:256], SUBSET["y_train"][:256], SUBSET["x_val"][:100],
                             SUBSET["y_val"][:100], **small_9)
        r2_9 = train_9.train(SUBSET["x_train"][:256], SUBSET["y_train"][:256], SUBSET["x_val"][:100],
                             SUBSET["y_val"][:100], **{**small_9, "seed": 1})
        r3_9 = train_9.train(SUBSET["x_train"][:256], SUBSET["y_train"][:256], SUBSET["x_val"][:100],
                             SUBSET["y_val"][:100], **small_9)
    keys_9 = {"model", "mean", "std", "history", "seconds"}
    if not (isinstance(r1_9, dict) and keys_9 <= set(r1_9)):
        verdict("B7.9", False, "", f"train doit rendre un dictionnaire avec les clés {sorted(keys_9)}, reçu "
                f"{type(r1_9).__name__} {sorted(r1_9) if isinstance(r1_9, dict) else ''}.")
        raise StopCheck
    h_9 = r1_9["history"]
    ok_9["history"] = (isinstance(h_9, dict) and all(len(h_9.get(k, [])) == 2 for k in ("train_loss", "val_loss", "val_acc"))
                       and all(type(v) is float for k in ("train_loss", "val_loss", "val_acc") for v in h_9[k])
                       and all(0 <= v <= 1 for v in h_9["val_acc"]))
    verdict("B7.9", ok_9["history"], "train sur 256 images, 2 epochs : un historique de deux floats par courbe.",
            f"train sur 256 images, 2 epochs : attendu history = {{train_loss, val_loss, val_acc}}, deux floats "
            f"chacune (val_acc dans [0, 1]), reçu {h_9!r:.200}.")
    stats_256_9 = (SUBSET["x_train"][:256] / 255.0).mean(axis=(0, 1, 2))
    ok_9["mean"] = np.allclose(r1_9["mean"], stats_256_9, atol=1e-10)
    verdict("B7.9", ok_9["mean"], "la normalisation est apprise sur les images d'entraînement reçues.",
            "result['mean'] doit être channel_stats des images d'entraînement reçues par train (ni la validation, ni "
            "les statistiques d'ImageNet).")
    ok_9["eval"] = not r1_9["model"].training
    verdict("B7.9", ok_9["eval"], "le modèle rendu est en mode eval().", "le modèle rendu est encore en mode "
            "entraînement : relis la docstring de train.")
    ok_9["seed"] = wbp.same_weights(r1_9["model"].state_dict(), r3_9["model"].state_dict()) and \
        not wbp.same_weights(r1_9["model"].state_dict(), r2_9["model"].state_dict())
    verdict("B7.9", ok_9["seed"], "deux appels avec seed=0 donnent les mêmes poids, seed=1 d'autres.",
            "deux appels de train avec seed=0 doivent donner exactement les mêmes poids, et seed=1 d'autres poids "
            "(seed_everything au début de train, le générateur de make_loader).")
    ok_9["print"] = printed_9.getvalue() == ""
    verdict("B7.9", ok_9["print"], "train n'écrit rien avec print.",
            f"train écrit avec print ({printed_9.getvalue().strip()[:80]!r}…) : passe par log.info et log.debug.")
    with wbp.torch_state_kept():
        TRAINED_9 = train_9.train(SUBSET["x_train"], SUBSET["y_train"], SUBSET["x_val"], SUBSET["y_val"])
    acc_9, s_9 = TRAINED_9["history"]["val_acc"][-1], TRAINED_9["seconds"]
    ok_9["smoke"] = s_9 < 60 and acc_9 > 0.15
    verdict("B7.9", ok_9["smoke"], f"entraînement fumée (réglages par défaut, 2 000 images, 1 epoch) : val_acc "
            f"{fr(acc_9, 3)} en {fr(s_9, 1)} s.", f"entraînement fumée (réglages par défaut, 2 000 images, 1 epoch) : "
            f"val_acc {fr(acc_9, 3)} en {fr(s_9, 1)} s ; attendu plus de 0,15 (le hasard : 0,10) en moins de 60 s.")

with checked("B7.9"):                                    # pip install -e, in a new environment
    if PYPROJECT_TOML is ...:
        print("⏳ Ex B7.9 : l'installation attend ton pyproject.toml.")
        raise StopCheck
    install_9 = wbp.install_editable(PROJECT)
    try:
        if install_9.ok is None:
            print("⏳ Ex B7.9 : ni pip ni uv dans cet environnement : l'installation éditable n'est pas vérifiée ici.")
            raise StopCheck
        ok_9["install"] = bool(install_9.ok)
        verdict("B7.9", install_9.ok, f"pip install -e . dans un environnement neuf ({install_9.method}, "
                f"{fr(install_9.seconds, 1)} s).", "pip install -e . échoue : " + install_9.log[-600:])
        if not install_9.ok:
            raise StopCheck
        probe_9 = ("import cifarprod, importlib.metadata as m, json; print(json.dumps([cifarprod.__file__, "
                   "m.version('cifarprod'), [e.value for e in m.entry_points(group='console_scripts') if e.name == "
                   "'cifarprod']]))")
        seen_9 = install_9.run(["python", "-c", probe_9], scratch("elsewhere"))
        where_9, version_9, entry_9 = json.loads(seen_9.stdout) if seen_9.ok else (None, None, None)
        ok_9["editable"] = seen_9.ok and Path(where_9).resolve().is_relative_to(PROJECT / "src")
        verdict("B7.9", ok_9["editable"], "depuis un autre dossier, import cifarprod lit ton dossier src/ (une "
                "installation éditable, pas une copie).", f"depuis un autre dossier, import cifarprod : {where_9 or seen_9.tail(1)}.")
        ok_9["metadata"] = version_9 == "0.1.0" and entry_9 == ["cifarprod.cli:main"]
        verdict("B7.9", ok_9["metadata"], "métadonnées installées : version 0.1.0, commande cifarprod → cifarprod.cli:main.",
                f"métadonnées installées : version {version_9!r} (attendu '0.1.0'), point d'entrée {entry_9!r} "
                "(attendu ['cifarprod.cli:main']).")
    finally:
        install_9.cleanup()
if ok_9 and all(ok_9.values()) and {"smoke", "editable", "metadata", "synthetic", "default"} <= set(ok_9):
    print("✅ Ex B7.9 : ton paquet s'installe, s'importe et s'entraîne : le notebook appelle, le paquet fait.")

### Ex B7.10 — Une interface en ligne de commande : argparse, logging et codes de sortie 📦 ★★ ⏱️ 30 min
**Objectif :** écrire la ligne de commande d'un entraînement : options, logging et codes de sortie qu'un autre programme sait lire.  
**Prérequis :** Ex B7.9 · fiche §207.2 (« argparse et le point d'entrée », « Le logging plutôt que print », « Les codes de sortie ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

**Le contrat.** Deux nouveaux fichiers ; la configuration, `config.py`, est fournie dans sa version de départ par la cellule suivante (une dataclass `TrainConfig` avec les réglages et leurs valeurs par défaut — `seed`, `epochs`, `lr`, `batch_size`, `n_train`, `arch`, `widths`, `data`, `deterministic`, `num_workers` —, et `load_config(path, overrides)`, sans aucune vérification des types ni des bornes : c'est B7.11). Elle n'est écrite que si `config.py` n'existe pas encore.

- `__main__.py` : `python -m cifarprod ...` appelle `main()` et transmet son code de sortie (207.2).
- `cli.py` : `build_parser()` rend un `ArgumentParser` de `prog="cifarprod"`, avec `--version` (« cifarprod 0.1.0 ») et la sous-commande **obligatoire** `train`, de quatre options : `--config FILE` (défaut `None` : les valeurs par défaut de `TrainConfig`), `--set KEY=VALUE` (répétable, défaut `[]`), `--out DIR` (défaut `runs/latest`) et `-v`/`--verbose`.
- `main(argv=None)` lit `argv` (`None` : `sys.argv`), configure le logging **une fois** (`logging.basicConfig`, niveau INFO, DEBUG avec `-v`, un format qui montre le niveau et le nom du logger : `%(levelname)s %(name)s: %(message)s`, sur stderr), puis : `load_config(args.config, args.set)` ; une `ValueError` ou une `OSError` → une ligne ERROR et le code **2** ; les parties `"train"` et `"val"` de `config.data` (un chemin relatif au dossier courant), dont il garde les `n_train` premières images d'entraînement ; des données illisibles ou trop peu d'images → une ligne ERROR et le code **1** ; `train(...)` avec les réglages de la configuration ; les poids dans `<out>/model.safetensors` (`safetensors.torch.save_file` du `state_dict`) ; une ligne INFO ; et sur stdout **une seule** ligne, le résumé JSON `{"status": "ok", "val_acc": <float>, "weights": <chemin absolu du fichier de poids>}` ; le code **0**. Toute autre exception pendant l'entraînement : `log.exception(...)` et le code 1. `main` **rend** son code (il n'appelle pas `sys.exit`, sauf argparse devant une mauvaise option, qui sort avec 2).

La cellule de vérification lance ta commande dans un autre processus (`python -m cifarprod`, depuis le dossier du projet, ton `src/` dans le `PYTHONPATH`) : `--version`, `--help`, sans sous-commande, avec une option inconnue, `--set lr` (sans `=`), des données absentes, un entraînement sur 200 images (son JSON, son fichier de poids, ses lignes INFO et l'absence de DEBUG), le même avec `-v` ; elle lit chaque fois le code de sortie, stdout et stderr. Puis elle appelle `main(["train", "--set", "lr"])` dans le notebook, qui doit rendre l'entier 2.

Dans tes notes : pourquoi le résumé va-t-il sur stdout et tout le reste sur stderr (pense à un script qui lit ce JSON) ? Pourquoi `main(argv)` reçoit-il ses arguments au lieu de lire `sys.argv`, et pourquoi rend-il son code au lieu d'appeler `sys.exit` ? Une intégration continue lance ta commande : que se passe-t-il si un entraînement qui échoue affiche son erreur mais rend 0 ?

In [ ]:
# the starting version of src/cifarprod/config.py, written by the check cell only if config.py does not exist yet (Ex B7.11 replaces it)
CONFIG_V0 = r'''"""The configuration of cifarprod, STARTING VERSION (given in Ex B7.10, replaced in Ex B7.11): the settings and their
defaults, read from a YAML file and the overrides "KEY=VALUE", without any check of the types or of the ranges."""

from __future__ import annotations

from dataclasses import asdict, dataclass, fields
from pathlib import Path

import yaml


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    epochs: int = 1
    lr: float = 0.05
    batch_size: int = 128
    n_train: int = 2000
    arch: str = "cnn"
    widths: tuple = (16, 32)
    data: str = "data/cifar10_2000.npz"
    deterministic: bool = True
    num_workers: int = 0

    def to_dict(self) -> dict:
        out = asdict(self)
        out["widths"] = list(self.widths)
        return out


def load_config(path=None, overrides=()):
    raw = dict(yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}) if path is not None else {}
    for item in overrides:
        key, sep, value = str(item).partition("=")
        if not sep:
            raise ValueError(f"--set expects KEY=VALUE, got {item!r}")
        raw[key.strip()] = yaml.safe_load(value)
    unknown = set(raw) - {f.name for f in fields(TrainConfig)}
    if unknown:
        raise ValueError(f"unknown settings: {sorted(unknown)}")
    return TrainConfig(**raw)
'''

In [ ]:
# Ex B7.10: two more files of your package (written by the check cell); config.py comes from the cell above

# src/cifarprod/__main__.py
MAIN_PY = r'''"""python -m cifarprod ...: the same command line as the `cifarprod` entry point."""

raise NotImplementedError("__main__.py")
'''

# src/cifarprod/cli.py (Ex B7.14 will ask you to complete this same cell)
CLI_PY = r'''"""The command line of cifarprod: cifarprod train [--config FILE] [--set KEY=VALUE] [--out DIR] [-v]; cifarprod --version."""

from __future__ import annotations

import argparse
import json
import logging
import sys
from pathlib import Path

from . import __version__
from .config import load_config

log = logging.getLogger(__name__)


def build_parser():
    """The parser of the contract: prog "cifarprod", --version, the sub-command train (required) and its options."""
    raise NotImplementedError("build_parser")


def main(argv=None):
    """Parse argv (None: sys.argv), configure the logging once, run the command and return its exit code: 0 success,
    2 invalid configuration (an ERROR line), 1 data that cannot be read or a training that fails (an ERROR line). On
    success, the only line written on stdout is the JSON summary {"status", "val_acc", "weights"}."""
    raise NotImplementedError("main")


if __name__ == "__main__":
    sys.exit(main())
'''

In [ ]:
ok_10 = {}
with checked("B7.10"):
    wbp.write_file(PROJECT, "src/cifarprod/config.py", CONFIG_V0, if_absent=True)
    files_10 = {"src/cifarprod/__main__.py": MAIN_PY, "src/cifarprod/cli.py": CLI_PY}
    if "TRACKING_PY" in globals() and TRACKING_PY is not ...:
        files_10["src/cifarprod/tracking.py"] = TRACKING_PY          # Ex B7.14, if it is already done
    wbp.write_files(PROJECT, files_10)
    cli_10 = wbp.import_module(PROJECT, "cli")
    cli_10.build_parser()                                 # ⏳ while it raises NotImplementedError
    out_10 = scratch("out")
    r_10 = wbp.cli(PROJECT, "--version")
    ok_10["version"] = run_line("B7.10", r_10, r_10.ok and "0.1.0" in r_10.stdout, "code 0, « cifarprod 0.1.0 ».",
                                "attendu le code 0 et la version 0.1.0 sur stdout")
    r_10 = wbp.cli(PROJECT, "--help")
    ok_10["help"] = run_line("B7.10", r_10, r_10.ok and "train" in r_10.stdout, "code 0, l'aide cite train.",
                             "attendu le code 0 et une aide qui cite la sous-commande train")
    r_10 = wbp.cli(PROJECT)
    ok_10["no command"] = run_line("B7.10", r_10, r_10.returncode == 2 and "usage" in r_10.stderr.lower(),
                                   "code 2 et l'usage sur stderr (la sous-commande est obligatoire).",
                                   "attendu le code 2 d'argparse : la sous-commande train est obligatoire")
    r_10 = wbp.cli(PROJECT, "train", "--epochs", "3")
    ok_10["unknown option"] = run_line("B7.10", r_10, r_10.returncode == 2, "code 2 (option inconnue).",
                                       "attendu le code 2 d'argparse pour une option inconnue")
    r_10 = wbp.cli(PROJECT, "train", "--set", "lr", "--out", out_10 / "a")
    ok_10["bad set"] = run_line("B7.10", r_10, r_10.returncode == 2 and r_10.stderr_lines("ERROR") and not r_10.stdout.strip(),
                                "code 2, une ligne ERROR, rien sur stdout.",
                                "attendu le code 2 (configuration invalide), une ligne ERROR sur stderr et rien sur stdout")
    r_10 = wbp.cli(PROJECT, "train", "--set", "data=data/absent.npz", "--out", out_10 / "b")
    ok_10["no data"] = run_line("B7.10", r_10, r_10.returncode == 1 and r_10.stderr_lines("ERROR") and not r_10.stdout.strip(),
                                "code 1, une ligne ERROR, rien sur stdout.",
                                "attendu le code 1 (données illisibles), une ligne ERROR sur stderr et rien sur stdout")
    r_10 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--out", out_10 / "c")
    lines_10 = [line for line in r_10.stdout.splitlines() if line.strip()]
    summary_10 = r_10.last_json()
    ok_json_10 = (len(lines_10) == 1 and isinstance(summary_10, dict) and summary_10.get("status") == "ok"
                  and isinstance(summary_10.get("val_acc"), float) and 0 <= summary_10["val_acc"] <= 1)
    ok_10["train"] = run_line("B7.10", r_10, r_10.ok and ok_json_10,
                              f"code 0, une ligne JSON sur stdout (val_acc {fr(summary_10['val_acc'], 3) if ok_json_10 else '?'}).",
                              "attendu le code 0 et UNE ligne JSON sur stdout, {\"status\": \"ok\", \"val_acc\": <float>, "
                              f"\"weights\": ...}} ; reçu {r_10.stdout.strip()[:200]!r}")
    if ok_10["train"]:
        weights_10 = Path(summary_10.get("weights", ""))
        from safetensors.torch import load_file
        ok_10["weights"] = (weights_10.is_absolute() and weights_10 == (out_10 / "c" / "model.safetensors").resolve()
                            and weights_10.is_file() and set(load_file(str(weights_10)))
                            == set(wbp.build_reference("cnn", (16, 32)).state_dict()))
        verdict("B7.10", ok_10["weights"], "les poids sont dans <out>/model.safetensors (chemin absolu dans le JSON), "
                "ceux du CNN (16, 32).", f"attendu \"weights\" = le chemin absolu de <out>/model.safetensors, un "
                f"fichier safetensors du CNN (16, 32) ; reçu {str(weights_10)!r}.")
        infos_10 = [line for line in r_10.stderr_lines("INFO") if "cifarprod.train" in line]
        ok_10["logging"] = bool(infos_10) and not r_10.stderr_lines("DEBUG")
        verdict("B7.10", ok_10["logging"], f"stderr : {len(r_10.stderr_lines('INFO'))} lignes INFO, dont "
                f"« {infos_10[-1].split(': ', 1)[-1][:60] if infos_10 else ''} » de cifarprod.train ; aucune DEBUG.",
                "attendu sur stderr des lignes INFO (au moins une du logger cifarprod.train, avec le niveau et le nom "
                f"du logger) et aucune ligne DEBUG sans -v ; reçu {r_10.tail(3)[:300]!r}")
        r_10 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--out", out_10 / "d", "-v")
        ok_10["verbose"] = run_line("B7.10", r_10, r_10.ok and any("cifarprod" in line for line in r_10.stderr_lines("DEBUG")),
                                    "avec -v, les lignes DEBUG des mini-batches.",
                                    "attendu, avec -v, des lignes DEBUG du paquet sur stderr")
    with wbp.isolated_logging():                         # main() configures the logging of THIS process
        try:
            code_10 = cli_10.main(["train", "--set", "lr"])
        except SystemExit as exit_10:                     # sys.exit inside main: the notebook would stop there
            code_10 = f"SystemExit({exit_10.code!r})"
    ok_10["main"] = type(code_10) is int and code_10 == 2
    verdict("B7.10", ok_10["main"], 'main(["train", "--set", "lr"]) rend l\'entier 2 dans le notebook.',
            f'main(["train", "--set", "lr"]) doit rendre l\'entier 2 (sans sys.exit) ; reçu {code_10!r}.')
if ok_10 and all(ok_10.values()):
    print("✅ Ex B7.10 : ta ligne de commande parle avec des codes de sortie et du logging.")

### Ex B7.11 — Configuration : dataclass + YAML, surcharges en ligne de commande, validation 📦 ★★ ⏱️ 30 min
**Objectif :** remplacer les réglages en dur par une configuration typée, validée au chargement et surchargeable.  
**Prérequis :** Ex B7.10 · fiche §207.3 (le code de `TrainConfig` et de `load_config`, « Rappel outil — YAML avec PyYAML ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

**Le contrat** de `config.py` (il remplace la version de départ de B7.10 ; `cli.py` ne change pas) :

- `TrainConfig` est une dataclass **figée** (`frozen=True`) aux dix champs et aux valeurs par défaut de la version de départ (`seed=0`, `epochs=1`, `lr=0.05`, `batch_size=128`, `n_train=2000`, `arch="cnn"`, `widths=(16, 32)`, `data="data/cifar10_2000.npz"`, `deterministic=True`, `num_workers=0`) ;
- à la construction (`__post_init__`), chaque réglage a **exactement** le type de sa valeur par défaut (`True` n'est pas un entier, la chaîne `"1e-3"` n'est pas un float), sauf deux conversions permises : un entier pour un float (`lr=1` devient `1.0`) et une liste pour un tuple (`widths=[8, 16]` devient `(8, 16)`) ; puis les bornes : `seed` ≥ 0, `epochs` ≥ 1, $0 <$ `lr` $< 1$, `batch_size` ≥ 1, `n_train` ≥ 1, `num_workers` ≥ 0, `arch` dans `("cnn", "vgg")`, `widths` non vide et fait d'entiers ≥ 1. Toute violation lève une `ValueError` dont le message **nomme le réglage** ;
- `to_dict()` rend un dictionnaire que `json.dumps` accepte (`widths` en liste) ;
- `load_config(path=None, overrides=())` part des valeurs par défaut, puis du fichier YAML `path` (`yaml.safe_load` ; un fichier vide donne les valeurs par défaut ; un fichier qui n'est pas un dictionnaire, `ValueError` ; un fichier absent, `FileNotFoundError`), puis des surcharges `"KEY=VALUE"` (chaque valeur lue par `yaml.safe_load`, donc `widths=[8, 16]` est une liste ; une surcharge sans `=`, `ValueError`) ; une clé inconnue, du fichier ou d'une surcharge, lève une `ValueError` qui la nomme.

Deux fichiers de configuration aussi : `configs/fast.yaml`, l'entraînement fumée du notebook, dont les réglages sont **exactement** les valeurs par défaut (écrits dans le fichier, pour qu'il se suffise) ; `configs/smoke.yaml`, l'essai de bout en bout de l'intégration continue : les mêmes, sauf `n_train: 256`, `batch_size: 64` et `widths: [8, 16]`.

La cellule de vérification essaie ta configuration dans le notebook : valeurs par défaut et `to_dict` en JSON, dataclass figée, conversions, seize surcharges invalides (chacune doit lever une `ValueError` qui nomme le réglage), l'ordre défauts → fichier → surcharges, trois fichiers pièges (`lr: 1e-3`, une faute de frappe, une liste), un fichier vide et un fichier absent, puis tes deux fichiers YAML ; enfin ta commande avec `--set epochs=0` (code 2, avant toute seconde de calcul) et `--config configs/smoke.yaml` (code 0).

Dans tes notes : que lit PyYAML dans `lr: 1e-3`, et pourquoi la validation des types est-elle la seule chose qui te sauve ici ? Pourquoi refuser une clé inconnue plutôt que l'ignorer ? Quels réglages ne mettrais-tu **jamais** dans ce fichier, et où iraient-ils (207.9) ?

In [ ]:
# Ex B7.11: the typed configuration (it replaces the starting version of B7.10) and two configuration files

# src/cifarprod/config.py
CONFIG_PY = r'''"""The configuration of cifarprod: a typed, frozen dataclass, read from a YAML file and overridden on the command line."""

from __future__ import annotations

from dataclasses import asdict, dataclass, fields
from pathlib import Path

import yaml


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    epochs: int = 1
    lr: float = 0.05
    batch_size: int = 128
    n_train: int = 2000
    arch: str = "cnn"
    widths: tuple = (16, 32)
    data: str = "data/cifar10_2000.npz"
    deterministic: bool = True
    num_workers: int = 0

    def __post_init__(self):
        """Check every setting (the contract of Ex B7.11): ValueError naming the setting."""
        raise NotImplementedError("TrainConfig.__post_init__")

    def to_dict(self):
        """The settings as a plain dictionary that json.dumps accepts (widths as a list)."""
        raise NotImplementedError("TrainConfig.to_dict")


def load_config(path=None, overrides=()):
    """The TrainConfig of the YAML file `path` (None: the defaults), then of the overrides "KEY=VALUE" (each VALUE read
    by yaml.safe_load). ValueError for a file that is not a mapping, an override without "=", an unknown key or a
    refused setting; FileNotFoundError for a missing file."""
    raise NotImplementedError("load_config")
'''

FAST_YAML = ...    # configs/fast.yaml: the smoke training of the notebook (the contract)
SMOKE_YAML = ...   # configs/smoke.yaml: the end-to-end test of the continuous integration (the contract)

In [ ]:
import dataclasses

ok_11 = {}
DEFAULTS_11 = DEFAULTS
with checked("B7.11"):
    wbp.write_files(PROJECT, {"src/cifarprod/config.py": CONFIG_PY, "configs/fast.yaml": FAST_YAML,
                              "configs/smoke.yaml": SMOKE_YAML})
    config_11 = wbp.import_module(PROJECT, "config")
    default_11 = config_11.TrainConfig()
    ok_11["defaults"] = default_11.to_dict() == DEFAULTS_11 and json.loads(json.dumps(default_11.to_dict())) == DEFAULTS_11
    verdict("B7.11", ok_11["defaults"], "TrainConfig() et to_dict() : les valeurs par défaut du contrat, en JSON.",
            f"TrainConfig().to_dict() doit valoir {DEFAULTS_11} ; reçu {default_11.to_dict()!r:.300}.")
    try:
        default_11.lr = 0.1
        ok_11["frozen"] = False
    except dataclasses.FrozenInstanceError:
        ok_11["frozen"] = True
    verdict("B7.11", ok_11["frozen"], "la configuration est figée (frozen=True).",
            "config.lr = 0.1 doit lever FrozenInstanceError : un run ne modifie pas sa configuration en route.")
    cast_11 = config_11.TrainConfig(lr=1 / 2 ** 4, epochs=2, widths=[8, 16])
    ok_11["cast"] = type(config_11.TrainConfig(seed=0, lr=0.5).lr) is float and cast_11.widths == (8, 16) \
        and type(config_11.TrainConfig(**{"lr": 1e-1}).lr) is float
    lr_int_11 = config_11.load_config(None, ["batch_size=64", "lr=0.25"])
    verdict("B7.11", ok_11["cast"] and lr_int_11.batch_size == 64, "un entier accepté pour un float, une liste YAML "
            "devient un tuple.", "attendu lr en float, widths=[8, 16] devenu (8, 16), et --set batch_size=64.")
    for override_11, name_11 in [("epochs=0", "epochs"), ("epochs=1.5", "epochs"), ("lr=0", "lr"), ("lr=1", "lr"),
                                 ("lr=1e-3", "lr"), ("batch_size=0", "batch_size"), ("batch_size=true", "batch_size"),
                                 ("n_train=0", "n_train"), ("seed=-1", "seed"), ("arch=resnet", "arch"),
                                 ("widths=[]", "widths"), ("widths=[8, 0]", "widths"), ("deterministic=yes2", "deterministic"),
                                 ("num_workers=-1", "num_workers"), ("lerning_rate=0.01", "lerning_rate"), ("lr", "lr")]:
        try:
            config_11.load_config(None, [override_11])
            ok_11[override_11] = False
            print(f"❌ Ex B7.11 : --set {override_11} : attendu ValueError, mais rien n'a été levé.")
        except ValueError as err_11:
            ok_11[override_11] = name_11 in str(err_11)
            if not ok_11[override_11]:
                print(f"❌ Ex B7.11 : --set {override_11} : ValueError levée, mais son message ne nomme pas {name_11!r} : "
                      f"« {str(err_11)[:120]} ».")
        except NotImplementedError:
            raise
        except Exception as err_11:  # noqa: BLE001 - the learner's code
            ok_11[override_11] = False
            print(f"❌ Ex B7.11 : --set {override_11} : attendu ValueError, reçu {type(err_11).__name__} : {str(err_11)[:120]}")
    refused_11 = [k for k in ok_11 if "=" in k or k == "lr"]
    if all(ok_11[k] for k in refused_11):
        print(f"✅ Ex B7.11 : {len(refused_11)} réglages invalides refusés, chacun par une ValueError qui le nomme "
              "(types, bornes, clé inconnue, --set sans =).")
    yaml_11 = scratch("yaml")
    (yaml_11 / "a.yaml").write_text("lr: 0.1\nepochs: 3\nwidths: [4, 8]\n", encoding="utf-8")
    both_11 = config_11.load_config(yaml_11 / "a.yaml", ["lr=0.01", "epochs=2"])
    ok_11["precedence"] = (both_11.lr, both_11.epochs, both_11.widths, both_11.seed) == (0.01, 2, (4, 8), 0)
    verdict("B7.11", ok_11["precedence"], "les valeurs par défaut, puis le fichier, puis les --set.",
            f"fichier lr 0.1, epochs 3, widths [4, 8] puis --set lr=0.01 epochs=2 : attendu (0.01, 2, (4, 8), seed 0), "
            f"reçu {(both_11.lr, both_11.epochs, both_11.widths, both_11.seed)}.")
    for text_11, what_11 in [("lr: 1e-3\n", "lr: 1e-3 (une chaîne pour PyYAML)"), ("lerning_rate: 0.01\n", "une faute de frappe"),
                             ("- lr\n- 0.1\n", "une liste au lieu d'un dictionnaire")]:
        (yaml_11 / "bad.yaml").write_text(text_11, encoding="utf-8")
        ok_11[what_11] = refuses("B7.11", f"un fichier avec {what_11}", lambda: config_11.load_config(yaml_11 / "bad.yaml"))
    (yaml_11 / "empty.yaml").write_text("# nothing\n", encoding="utf-8")
    ok_11["empty"] = config_11.load_config(yaml_11 / "empty.yaml").to_dict() == DEFAULTS_11
    verdict("B7.11", ok_11["empty"], "un fichier vide : les valeurs par défaut.", "un fichier vide (un commentaire) doit "
            "donner les valeurs par défaut (yaml.safe_load rend None).")
    ok_11["absent"] = refuses("B7.11", "un fichier absent", lambda: config_11.load_config(yaml_11 / "absent.yaml"),
                              error=FileNotFoundError)
    want_files_11 = {"configs/fast.yaml": DEFAULTS_11,
                     "configs/smoke.yaml": {**DEFAULTS_11, "n_train": 256, "batch_size": 64, "widths": [8, 16]}}
    for rel_11, want_11 in want_files_11.items():
        got_11 = config_11.load_config(PROJECT / rel_11).to_dict()
        ok_11[rel_11] = got_11 == want_11
        verdict("B7.11", ok_11[rel_11], f"{rel_11} : les réglages du contrat.",
                f"{rel_11} : attendu {want_11}, reçu {got_11}.")
    out_11 = scratch("out")
    r_11 = wbp.cli(PROJECT, "train", "--set", "epochs=0", "--out", out_11 / "a")
    ok_11["cli epochs"] = run_line("B7.11", r_11, r_11.returncode == 2 and any("epochs" in l for l in r_11.stderr_lines("ERROR")),
                                   "code 2 et une ligne ERROR qui nomme epochs, avant toute seconde de calcul.",
                                   "attendu le code 2 et une ligne ERROR qui nomme epochs")
    r_11 = wbp.cli(PROJECT, "train", "--config", "configs/smoke.yaml", "--out", out_11 / "b")
    ok_11["cli smoke"] = run_line("B7.11", r_11, r_11.ok, f"code 0 en {fr(r_11.seconds, 1)} s.", "attendu le code 0")
if ok_11 and all(ok_11.values()):
    print("✅ Ex B7.11 : ta configuration est typée, validée au chargement et surchargeable.")

### Ex B7.12 — Prédire : même graine, puis num_workers = 4, puis GPU — résultats identiques ? 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que les workers d'un DataLoader et un GPU changent à un entraînement à graine fixée.  
**Prérequis :** Ex B7.11 · ch. 24 (workers) · fiche §207.4 (« Les graines et le déterminisme ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

L'expérience entraîne six fois un petit CNN (`wbp.workers_run`, 512 images fixes, 2 epochs, graine 0) et compare les poids **au bit près** (`torch.equal` sur chaque tenseur). Chaque run fait `torch.manual_seed(0)`, puis lit ses images par un `DataLoader(..., shuffle=True, generator=torch.Generator().manual_seed(0), num_workers=...)`, sans `worker_init_fn` ni `persistent_workers`. Le dataset applique une **augmentation** : dans son `__getitem__`, il retourne l'image (gauche-droite) quand `torch.rand(1) < 0.5`. Avec `num_workers = 0`, `__getitem__` tourne dans le processus du notebook ; avec `num_workers = 4`, dans quatre processus fils (chaque epoch en lance de nouveaux). Avec 4 workers sur une machine à 2 cœurs, PyTorch avertit qu'il en propose au plus 2 : l'avertissement est tu, il ne change rien aux nombres. **Avant d'exécuter**, écris tes quatre prédictions (`True` ou `False`) :
- a) `replay0_12` : deux runs avec `num_workers = 0` donnent-ils exactement les mêmes poids ?
- b) `replay4_12` : deux runs avec `num_workers = 4` donnent-ils exactement les mêmes poids ?
- c) `zero_four_12` : le run à `num_workers = 0` et le run à `num_workers = 4` donnent-ils exactement les mêmes poids ?
- d) `plain_12` : **sans** l'augmentation (aucun tirage dans `__getitem__`), `num_workers = 0` et `num_workers = 4` donnent-ils exactement les mêmes poids ?

Et, dans ta cellule 📝 seulement (aucune vérification) : e) sur un GPU, avec la même graine et `num_workers = 0`, les poids seraient-ils ceux du CPU ? Ceux d'un second run sur le même GPU ? La cellule d'expérience le mesure si un GPU est là (sur Colab), et le dit sinon.

La cellule d'expérience n'affiche rien tant que tes quatre prédictions ne sont pas écrites ; elle les vérifie à la fin. Rien n'y dépend de ton paquet.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
replay0_12 = ...     # a) two runs, seed 0, num_workers=0: exactly the same weights? True or False
replay4_12 = ...     # b) two runs, seed 0, num_workers=4: exactly the same weights? True or False
zero_four_12 = ...   # c) the run with num_workers=0 and the run with num_workers=4: the same weights? True or False
plain_12 = ...       # d) WITHOUT the random flip, num_workers=0 against num_workers=4: the same weights? True or False

In [ ]:
if any(answer is ... or answer is None for answer in [replay0_12, replay4_12, zero_four_12, plain_12]):
    print("⏳ Ex B7.12 : écris d'abord tes quatre prédictions dans la cellule précédente.")
else:
    start_12 = time.perf_counter()
    RUNS_12 = {"0, essai 1": wbp.workers_run(0, seed=0), "0, essai 2": wbp.workers_run(0, seed=0),
               "4, essai 1": wbp.workers_run(4, seed=0), "4, essai 2": wbp.workers_run(4, seed=0),
               "0, sans retournement": wbp.workers_run(0, seed=0, augment=False),
               "4, sans retournement": wbp.workers_run(4, seed=0, augment=False)}
    PAIRS_12 = {"a": ("0, essai 1", "0, essai 2"), "b": ("4, essai 1", "4, essai 2"), "c": ("0, essai 1", "4, essai 1"),
                "d": ("0, sans retournement", "4, sans retournement")}
    print(f"Six entraînements courts en {fr(time.perf_counter() - start_12, 1)} s (512 images, 2 epochs, graine 0) :")
    for letter_12, (first_12, second_12) in PAIRS_12.items():
        a_12, b_12 = RUNS_12[first_12], RUNS_12[second_12]
        print(f"  {letter_12}) num_workers {first_12} contre {second_12} : "
              + ("mêmes poids au bit près" if wbp.same_weights(a_12, b_12) else f"poids différents (écart maximal "
                 f"{wbp.max_gap(a_12, b_12):.2e})"))
    if torch.cuda.is_available():
        gpu_12 = [wbp.workers_run(0, seed=0, device="cuda") for _ in range(2)]
        print(f"  e) sur le GPU ({torch.cuda.get_device_name(0)}) : deux runs "
              + ("identiques" if wbp.same_weights(*gpu_12) else f"différents ({wbp.max_gap(*gpu_12):.2e})")
              + " ; contre le CPU : " + ("identiques" if wbp.same_weights(gpu_12[0], RUNS_12["0, essai 1"])
                                         else f"différents ({wbp.max_gap(gpu_12[0], RUNS_12['0, essai 1']):.2e})"))
    else:
        print("  e) pas de GPU ici : ta prédiction e) se compare à 05_solutions.md (ou relance cette cellule sur un GPU "
              "de Colab).")
    wb.check("B7.12a", replay0_12)
    wb.check("B7.12b", replay4_12)
    wb.check("B7.12c", zero_four_12)
    wb.check("B7.12d", plain_12)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? D'où chaque worker tient-il son générateur, et pourquoi la graine du `DataLoader` suffit-elle à le rejouer ? Que faudrait-il pour qu'un run à 0 worker et un run à 4 workers tirent les mêmes retournements ? Que ferait `worker_init_fn` (207.4) pour les tirages de NumPy et de `random` dans un worker ? Pourquoi ces résultats ne disent-ils rien d'un GPU ?

### Ex B7.13 — Mesurer le non-déterminisme : graines fixées ou non, algorithmes déterministes, cinq exécutions 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer, sur ton propre entraînement, ce que changent la graine et les algorithmes déterministes, et rapporter la dispersion d'un score.  
**Prérequis :** Ex B7.12 · ✏️ B7.2 · fiche §207.4 (« Les graines et le déterminisme », « Rapporter un score ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

Tu lances **ta** commande cinq fois et tu compares les poids qu'elle écrit. **Contrat** des trois fonctions de la cellule suivante :
- `train_run_13(seed, deterministic=True)` lance une fois, par `wbp.cli(PROJECT, ...)`, la commande `train --config configs/fast.yaml --set seed=<seed> --set deterministic=<true|false> --out <un nouveau dossier temporaire>` ; elle rend `{"weights": le dictionnaire de tenseurs du fichier de poids écrit, "val_acc": la val_acc de la ligne JSON, "seconds": la durée de la commande}`, et lève une `RuntimeError` avec la fin de stderr si le code de sortie n'est pas 0 ;
- `same_weights_13(a, b)` rend `True` quand les deux dictionnaires ont les mêmes noms et exactement les mêmes tenseurs, `False` sinon (un booléen) ;
- `max_gap_13(a, b)` rend le plus grand écart absolu entre les tenseurs de même nom, un float Python.

La cellule de vérification essaie `same_weights_13` et `max_gap_13` sur trois petits cas (des copies, un poids qui diffère de $10^{-6}$, un tenseur en moins). L'expérience lance ensuite cinq runs de 2 000 images (≈ 4 s chacun) : A et B, graine 0 ; C, graine 0 sans les algorithmes déterministes ; D et E, graines 1 et 2. Elle affiche les écarts A-B, A-C, A-D et D-E, l'accuracy des graines 0, 1 et 2 avec son intervalle à 95 % (la formule de ✏️ B7.2, $t_{0{,}975;\,2} = 4{,}303$, donné par le tableau de 207.4), et vérifie par `wb.check` si A et B ont exactement les mêmes poids (rien à taper : la valeur vient de ton code).

Dans tes notes : que donne A contre C sur ce processeur, et pourquoi ce résultat ne vaut-il que pour **ce** matériel et **ce** logiciel (207.4) ? De quel ordre est l'écart d'une graine à l'autre, et l'écart d'accuracy ? Avec trois graines, que vaut l'intervalle, et que dirais-tu d'un concurrent annoncé 1 point au-dessus sur une seule graine ? Qu'est-ce qui, dans ton paquet, rend A et B identiques : liste les générateurs que la graine fixe.

In [ ]:
def train_run_13(seed, deterministic=True):
    """Run YOUR command line once in PROJECT: python -m cifarprod train --config configs/fast.yaml --set seed=<seed>
    --set deterministic=<true|false> --out <a new temporary folder>, through wbp.cli. Returns {"weights": the dict of
    tensors of the model.safetensors it wrote, "val_acc": the val_acc of its JSON line, "seconds": its duration};
    RuntimeError with the end of stderr if the command does not end with the code 0."""
    raise NotImplementedError("train_run_13")


def same_weights_13(a, b):
    """True when the two dicts of tensors have the same names and exactly the same values (torch.equal), else False."""
    raise NotImplementedError("same_weights_13")


def max_gap_13(a, b):
    """The largest absolute difference between the tensors of the same name of two dicts, as a Python float."""
    raise NotImplementedError("max_gap_13")

In [ ]:
tools_ok_13 = False
with checked("B7.13"):
    base_13 = {"w": torch.zeros(2, 3), "b": torch.ones(4), "n": torch.tensor(7)}
    twin_13 = {k: v.clone() for k, v in base_13.items()}
    tiny_13 = {**twin_13, "w": torch.tensor([[0.0, 0.0, 1e-6], [0.0, 0.0, 0.0]])}
    cases_13 = [("deux copies identiques", base_13, twin_13, True), ("un poids qui diffère de 1e-6", base_13, tiny_13, False),
                ("un tenseur en moins", base_13, {k: v for k, v in twin_13.items() if k != "b"}, False)]
    all_13 = True
    for label_13, a_13, b_13, want_13 in cases_13:
        got_13 = same_weights_13(a_13, b_13)
        good_13 = isinstance(got_13, (bool, np.bool_)) and bool(got_13) == want_13
        all_13 &= good_13
        verdict("B7.13", good_13, f"same_weights_13 sur {label_13} : {want_13}.",
                f"same_weights_13 sur {label_13} : attendu {want_13}, reçu {got_13!r}.")
    gap_13 = max_gap_13(base_13, tiny_13)
    ok_gap_13 = type(gap_13) is float and abs(gap_13 - float(torch.tensor(1e-6))) < 1e-12
    verdict("B7.13", ok_gap_13, "max_gap_13 : 1e-6 (un float Python).",
            f"max_gap_13 sur un poids qui diffère de 1e-6 : reçu {gap_13!r}, attendu le float 1e-6 (à l'arrondi du float32 près).")
    tools_ok_13 = bool(all_13 and ok_gap_13)
probe_13 = wbp.cli(PROJECT, "--version")
if tools_ok_13 and not probe_13.ok:
    tools_ok_13 = False
    print(f"⏳ Ex B7.13 : ta commande ne répond pas encore (cifarprod --version : code {probe_13.returncode}, "
          f"{probe_13.tail(1)[:160]}) : fais d'abord passer B7.10 et B7.11.")

In [ ]:
RUNS_13 = {}
if not tools_ok_13:
    print("⏳ Ex B7.13 : l'expérience attend same_weights_13, max_gap_13 et train_run_13 justes (cellule précédente).")
else:
    with checked("B7.13"):
        PLAN_13 = {"A": (0, True), "B": (0, True), "C": (0, False), "D": (1, True), "E": (2, True)}
        for key_13, (seed_13, det_13) in PLAN_13.items():
            RUNS_13[key_13] = train_run_13(seed_13, det_13)
            print(f"run {key_13} : graine {seed_13}, algorithmes déterministes {'oui' if det_13 else 'non'} → val_acc "
                  f"{fr(RUNS_13[key_13]['val_acc'], 3)} en {fr(RUNS_13[key_13]['seconds'], 1)} s")
        for first_13, second_13, what_13 in [("A", "B", "même graine, deux fois"), ("A", "C", "déterministe ou non"),
                                             ("A", "D", "graine 0 contre graine 1"), ("D", "E", "graine 1 contre graine 2")]:
            wa_13, wb_13 = RUNS_13[first_13]["weights"], RUNS_13[second_13]["weights"]
            print(f"  {first_13} contre {second_13} ({what_13}) : " + ("poids identiques au bit près"
                  if same_weights_13(wa_13, wb_13) else f"écart maximal {max_gap_13(wa_13, wb_13):.3g}"))
        accs_13 = np.array([RUNS_13[k]["val_acc"] for k in ("A", "D", "E")])
        half_13 = 4.303 * accs_13.std(ddof=1) / np.sqrt(3)                # t(0.975; 2), the table of 207.4
        print(f"accuracy sur 3 graines : {fr(100 * accs_13.mean(), 1)} % ± {fr(100 * half_13, 1)} (intervalle à 95 %, "
              f"t = 4,303 pour n = 3 ; valeurs {', '.join(fr(100 * a, 1) for a in accs_13)})")
        wb.check("B7.13", same_weights_13(RUNS_13["A"]["weights"], RUNS_13["B"]["weights"]))

### Ex B7.14 — Suivi d'expériences maison : identifiant de run, configuration, métriques et artefacts en JSONL 📦 ★★ ⏱️ 30 min
**Objectif :** faire laisser à chaque run une ligne qui dit ce qui le définit, ce qu'il mesure et ce qu'il produit.  
**Prérequis :** Ex B7.11 · ch. 24 (journal d'expériences) · 🔁 R1 · fiche §207.4 (versions, empreinte), §207.5 · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

> **Mode d'emploi du paquet** : chaque fichier de ton paquet est une chaîne de la cellule à compléter (`DATA_PY = r'''...'''`) ; la cellule de vérification l'**écrit** dans ton projet (`mon_travail/b7_production/cifar-prod/`, à côté de ce notebook), puis l'importe ou le lance dans un autre processus. Pour corriger un fichier, corrige sa chaîne et réexécute les deux cellules. Si tu préfères un éditeur, modifie le fichier sur le disque : la cellule le voit et ne l'écrase plus (elle le dit).

**Le contrat.** Un nouveau fichier, `tracking.py` (la cellule suivante : exécute-la d'abord), et `cli.py` à **compléter dans la cellule de B7.10** ; réexécute ensuite la cellule de vérification de B7.10 (qui écrit aussi `tracking.py`, et vérifie que rien n'a cassé), puis celle-ci :
- `tracking.py` : `new_run_id()` (la date et l'heure UTC puis six caractères hexadécimaux au hasard : `20261104-161731-e06339`, comme 207.5) ; `file_sha256(path)` ; `artifact(path)` rend `{"path": str(path), "bytes": sa taille, "sha256": file_sha256(path)}` ; `environment()` rend `{"python", "numpy", "torch", "platform", "commit", "dirty"}` (`platform.python_version()`, `importlib.metadata.version(...)`, `platform.platform()`, le commit court `git rev-parse --short HEAD` et si `git status --porcelain` n'est pas vide, `None` hors d'un dépôt git) ; `log_run(path, record)` ajoute **une** ligne JSON (clés triées) à la fin du journal, dont il crée le dossier ; `read_runs(path)` rend la liste des runs, du plus ancien au plus récent (`[]` sans fichier) ;
- `cli.py` : une option de plus, `--runs FILE` (défaut `runs/runs.jsonl`) ; sans `--out`, les poids vont dans `runs/<run_id>/` ; **chaque** run qui a commencé (après une configuration valide) laisse une ligne, réussi ou non (un `finally`), avec les champs `run_id`, `started` (la date ISO 8601 UTC), `status` (`"ok"` ou `"failed"`), `seconds`, `config` (`to_dict()`), `environment` ; un run réussi ajoute `data` (`{"path", "n_train", "sha256"}` : l'empreinte `fingerprint(X, y)` des images **et** des labels d'entraînement utilisés), `metrics` (les trois courbes de `history` et `val_acc_final`), `normalization` (`{"mean", "std"}`) et `artifacts` (la liste des `artifact(...)` des fichiers écrits : ici, les poids) ; un run raté garde son `error` (le type et le message de l'exception) ; le résumé JSON de stdout ajoute le `run_id`.

La cellule de vérification essaie `tracking.py` dans le notebook (trois identifiants, l'empreinte d'un fichier de 300 000 octets contre `hashlib`, l'environnement, deux lignes ajoutées puis relues), puis lance trois runs dans un même journal : 200 images, puis `n_train=99999` (plus que les données : code 1), puis 200 images avec la graine 1 sans `--out`. Elle relit le journal (trois lignes, la première jamais réécrite) et compare chaque champ à ce qu'elle sait : la configuration demandée, l'empreinte des 200 premières images et de leurs labels, le fichier de poids (taille et SHA-256), la val_acc de stdout, l'erreur du run raté ; elle affiche enfin le journal par `pandas.json_normalize`.

Dans tes notes : six mois plus tard, quelle ligne du journal te dit avec quelles données, quel code et quels réglages le modèle livré a été entraîné ? Pourquoi garder les runs ratés ? Pourquoi l'empreinte porte-t-elle aussi sur les labels ? Que manque-t-il encore pour rejouer un run sur une autre machine (207.4) ?

In [ ]:
# Ex B7.14: src/cifarprod/tracking.py; then complete CLI_PY in the cell of Ex B7.10 (the contract) and run that cell again
TRACKING_PY = r'''"""The run journal of cifarprod: one JSON line per run, appended to a JSONL file, never rewritten."""

from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import uuid
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path


def new_run_id():
    """A unique run identifier: the UTC date and time, then six random hexadecimal characters
    ("20261104-161731-e06339")."""
    raise NotImplementedError("new_run_id")


def file_sha256(path):
    """The SHA-256 of the bytes of a file (64 hexadecimal characters)."""
    raise NotImplementedError("file_sha256")


def artifact(path):
    """{"path": str(path), "bytes": its size, "sha256": file_sha256(path)} of a file written by the run."""
    raise NotImplementedError("artifact")


def environment(cwd=None):
    """{"python", "numpy", "torch", "platform", "commit", "dirty"}: the versions (platform.python_version(),
    importlib.metadata.version), platform.platform(), the short git commit and whether `git status --porcelain` is not
    empty (None and None outside a git repository)."""
    raise NotImplementedError("environment")


def log_run(path, record):
    """Append one JSON line (keys sorted) to the journal `path`; its folder is created if needed."""
    raise NotImplementedError("log_run")


def read_runs(path):
    """The runs of the journal as a list of dicts, oldest first ([] if the file does not exist)."""
    raise NotImplementedError("read_runs")
'''

In [ ]:
import hashlib
import platform
from importlib import metadata

import pandas as pd

ok_14 = {}
with checked("B7.14"):
    wbp.write_files(PROJECT, {"src/cifarprod/tracking.py": TRACKING_PY, "src/cifarprod/cli.py": CLI_PY})
    tracking_14 = wbp.import_module(PROJECT, "tracking")
    ids_14 = [tracking_14.new_run_id() for _ in range(3)]
    today_14 = time.strftime("%Y%m%d", time.gmtime())
    ok_14["id"] = (all(isinstance(i, str) and re.fullmatch(r"\d{8}-\d{6}-[0-9a-f]{6}", i) for i in ids_14)
                   and len(set(ids_14)) == 3 and ids_14[0][:8] == today_14)
    verdict("B7.14", ok_14["id"], f"new_run_id : {ids_14[0]}, {ids_14[1]}… (date UTC, heure, six caractères au hasard).",
            f"new_run_id : attendu trois identifiants distincts de la forme {today_14}-HHMMSS-xxxxxx (date UTC), "
            f"reçu {ids_14!r:.120}.")
    blob_14 = scratch("blob") / "weights.bin"
    blob_14.write_bytes(np.random.default_rng(0).bytes(300_000))
    want_art_14 = {"path": str(blob_14), "bytes": 300_000, "sha256": hashlib.sha256(blob_14.read_bytes()).hexdigest()}
    ok_14["artifact"] = tracking_14.file_sha256(blob_14) == want_art_14["sha256"] and tracking_14.artifact(blob_14) == want_art_14
    verdict("B7.14", ok_14["artifact"], "file_sha256 et artifact d'un fichier de 300 000 octets : comme hashlib.",
            f"artifact(fichier de 300 000 octets) : attendu {{'path', 'bytes', 'sha256'}} = {want_art_14['bytes']} octets "
            f"et le SHA-256 de hashlib, reçu {tracking_14.artifact(blob_14)!r:.200}.")
    env_14 = tracking_14.environment()
    ok_14["environment"] = (isinstance(env_14, dict) and {"python", "numpy", "torch", "platform", "commit", "dirty"} <= set(env_14)
                            and env_14["python"] == platform.python_version() and env_14["torch"] == metadata.version("torch"))
    verdict("B7.14", ok_14["environment"], f"environment : Python {env_14.get('python')}, torch {env_14.get('torch')}, "
            f"commit {env_14.get('commit')!r}.", "environment doit rendre {python, numpy, torch, platform, commit, dirty} "
            f"avec platform.python_version() et importlib.metadata.version ; reçu {env_14!r:.200}.")
    journal_14 = scratch("journal") / "sub" / "runs.jsonl"
    tracking_14.log_run(journal_14, {"run_id": "a", "config": {"lr": 0.1}})
    first_14 = journal_14.read_text(encoding="utf-8")
    tracking_14.log_run(journal_14, {"run_id": "b", "metrics": [1, 2]})
    lines_14 = journal_14.read_text(encoding="utf-8").splitlines()
    ok_14["append"] = (len(lines_14) == 2 and journal_14.read_text(encoding="utf-8").startswith(first_14)
                       and [json.loads(l) for l in lines_14] == [{"run_id": "a", "config": {"lr": 0.1}}, {"run_id": "b", "metrics": [1, 2]}]
                       and tracking_14.read_runs(journal_14) == [json.loads(l) for l in lines_14]
                       and tracking_14.read_runs(journal_14.with_name("absent.jsonl")) == [])
    verdict("B7.14", ok_14["append"], "log_run ajoute une ligne JSON par run (dossier créé), read_runs les relit ([] sans fichier).",
            f"log_run puis read_runs : attendu deux lignes JSON, la première inchangée, relues dans l'ordre, et [] pour "
            f"un journal absent ; le fichier contient {lines_14!r:.200}.")

ready_14 = (written(lambda: wbp.import_module(PROJECT, "tracking").new_run_id())
            and written(lambda: wbp.import_module(PROJECT, "cli").build_parser())
            and written(lambda: wbp.import_module(PROJECT, "config").TrainConfig()))
if not ready_14:
    print("⏳ Ex B7.14 : le journal des runs attend tracking.py, et ta commande et ta configuration (B7.10, B7.11).")
else:
    with checked("B7.14"):                                 # the command line logs every run
        out_14 = scratch("runs")
        journal_14 = out_14 / "runs.jsonl"
        r1_14 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--out", out_14 / "a", "--runs", journal_14)
        ok_14["run 1"] = run_line("B7.14", r1_14, r1_14.ok, "code 0.", "attendu le code 0")
        first_lines_14 = journal_14.read_text(encoding="utf-8").splitlines() if journal_14.is_file() else []
        first_line_14 = first_lines_14[0] if first_lines_14 else None
        r2_14 = wbp.cli(PROJECT, "train", "--set", "n_train=99999", "--runs", journal_14)
        ok_14["run 2"] = run_line("B7.14", r2_14, r2_14.returncode == 1, "code 1 (n_train plus grand que les données).",
                                  "attendu le code 1 : n_train = 99 999 dépasse les 2 000 images")
        r3_14 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--set", "seed=1", "--runs", journal_14)
        ok_14["run 3"] = run_line("B7.14", r3_14, r3_14.ok, "code 0, sans --out.", "attendu le code 0")
        lines_14 = journal_14.read_text(encoding="utf-8").splitlines() if journal_14.is_file() else []
        runs_14 = []
        for line_14 in lines_14:
            try:
                record_14 = json.loads(line_14)
            except ValueError:
                record_14 = None
            runs_14.append(record_14 if isinstance(record_14, dict) else {"status": f"ligne illisible : {line_14[:60]!r}"})
        ok_14["three"] = [r.get("status") for r in runs_14] == ["ok", "failed", "ok"] and lines_14[0] == first_line_14
        verdict("B7.14", ok_14["three"], "trois lignes dans le journal, statuts ok, failed, ok ; la première jamais réécrite.",
                f"attendu trois lignes de statuts ['ok', 'failed', 'ok'], la première inchangée ; reçu "
                f"{[r.get('status') for r in runs_14]}.")
        if not ok_14["three"]:
            raise StopCheck
        keys_14 = {"run_id", "started", "status", "seconds", "config", "environment", "data", "metrics", "normalization",
                   "artifacts"}
        ok_rec_14 = all(keys_14 <= set(r) for r in (runs_14[0], runs_14[2])) and {"run_id", "status", "seconds", "config",
                                                                                   "error"} <= set(runs_14[1])
        verdict("B7.14", ok_rec_14, "chaque ligne a les champs du contrat (un run raté : son erreur).",
                f"champs manquants : run réussi {sorted(keys_14 - set(runs_14[0]))}, run raté "
                f"{sorted({'run_id', 'status', 'seconds', 'config', 'error'} - set(runs_14[1]))}.")
        if not ok_rec_14:
            raise StopCheck
        rec_14 = runs_14[0]
        ok_14["ids"] = len({r["run_id"] for r in runs_14}) == 3 and r1_14.last_json().get("run_id") == rec_14["run_id"] \
            and r3_14.last_json().get("run_id") == runs_14[2]["run_id"]
        verdict("B7.14", ok_14["ids"], "trois identifiants distincts, ceux des lignes JSON de stdout.",
                "attendu trois run_id distincts, et sur stdout le run_id de la ligne du journal.")
        ok_14["config"] = rec_14["config"] == {**DEFAULTS, "n_train": 200} and runs_14[2]["config"] == {**DEFAULTS, "n_train": 200, "seed": 1}
        verdict("B7.14", ok_14["config"], "la configuration complète du run (to_dict), surcharges comprises.",
                f"config du premier run : attendu {{**DEFAULTS, 'n_train': 200}}, reçu {rec_14['config']!r:.200}.")
        want_sha_14 = wbp.array_fingerprint(SUBSET["x_train"][:200], SUBSET["y_train"][:200])
        ok_14["data"] = rec_14["data"].get("sha256") == want_sha_14 and rec_14["data"].get("n_train") == 200
        verdict("B7.14", ok_14["data"], f"l'empreinte des 200 images d'entraînement et de leurs labels : {want_sha_14[:12]}….",
                f"data : attendu n_train 200 et sha256 = fingerprint(X[:200], y[:200]) des images ET des labels utilisés, "
                f"reçu {rec_14['data']!r:.200}.")
        arts_14 = rec_14["artifacts"]
        art_path_14 = (PROJECT / arts_14[0]["path"]).resolve() if arts_14 else None
        ok_14["artifacts"] = (len(arts_14) >= 1 and art_path_14 == (out_14 / "a" / "model.safetensors").resolve()
                              and arts_14[0].get("sha256") == hashlib.sha256(art_path_14.read_bytes()).hexdigest()
                              and arts_14[0].get("bytes") == art_path_14.stat().st_size)
        verdict("B7.14", ok_14["artifacts"], "artifacts : le fichier des poids, sa taille et son SHA-256.",
                f"artifacts : attendu [{{path, bytes, sha256}}] du fichier <out>/model.safetensors, reçu {arts_14!r:.200}.")
        default_out_14 = (PROJECT / runs_14[2]["artifacts"][0]["path"]).resolve() if runs_14[2]["artifacts"] else None
        ok_14["default out"] = default_out_14 == (PROJECT / "runs" / runs_14[2]["run_id"] / "model.safetensors").resolve()
        verdict("B7.14", ok_14["default out"], "sans --out, les poids vont dans runs/<run_id>/.",
                f"sans --out, attendu runs/<run_id>/model.safetensors, reçu {default_out_14}.")
        metrics_14 = rec_14["metrics"]
        ok_14["metrics"] = (len(metrics_14.get("val_acc", [])) == 1 and metrics_14.get("val_acc_final") == r1_14.last_json().get("val_acc")
                            and np.allclose(rec_14["normalization"]["mean"], (SUBSET["x_train"][:200] / 255).mean(axis=(0, 1, 2))))
        verdict("B7.14", ok_14["metrics"], "metrics (une valeur par epoch, val_acc_final) et la normalisation apprise.",
                "metrics : attendu train_loss, val_loss, val_acc (une valeur par epoch) et val_acc_final, la val_acc de "
                "stdout ; normalization : la moyenne et l'écart-type des images d'entraînement.")
        ok_14["failed"] = "n_train" in runs_14[1].get("error", "")
        verdict("B7.14", ok_14["failed"], f"le run raté garde son erreur : « {runs_14[1].get('error', '')[:70]} ».",
                f"le run raté doit garder son erreur (qui nomme n_train) ; reçu {runs_14[1].get('error')!r}.")
        TABLE_14 = pd.json_normalize(runs_14)[["run_id", "status", "config.seed", "config.n_train", "metrics.val_acc_final",
                                               "seconds"]]
        print(TABLE_14.to_string(index=False))
if ok_14 and all(ok_14.values()) and len(ok_14) >= 12:
    print("✅ Ex B7.14 : chaque run laisse sa ligne : ce qui le définit, ce qu'il mesure, ce qu'il produit.")

### Ex B7.15 — Tests pytest pour le ML : formes, overfitting d'un mini-batch, prétraitement, schéma des données 📦 ★★★ ⏱️ 45 min
**Objectif :** écrire les tests qui protègent un projet de ML, et vérifier qu'ils attrapent de vraies erreurs.  
**Prérequis :** Ex B7.11 (et B7.14) · ch. 21 (tests paramétrés, 21.16) · fiche §207.6 (« Quatre familles de tests ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

La fiche donne en entier deux tests du modèle et sa fixture : ils sont dans `TEST_MODEL_PY`, avec les fixtures de `CONFTEST_PY` (`model`, le CNN de la fiche ; `data_file`, le chemin de `data/cifar10_2000.npz`, qui fait sauter le test sans données, comme dans l'intégration continue ; `synthetic_file`, un petit `.npz` au même schéma, 96 images d'entraînement et 32 de validation). **À toi** `tests/test_data.py` et `tests/test_cli.py`, avec **au moins 8 tests** (les cas d'un `parametrize` comptent chacun) dans ces deux fichiers, qui couvrent :
- le **prétraitement** d'une image constante (tout à 0, tout à 255 : la valeur attendue de chaque canal se calcule à la main) et l'ordre des canaux ;
- le **schéma des données** : types, formes, labels de 0 à 9, chaque classe présente, et le refus d'un fichier mal formé ;
- **aucun recouvrement** entre l'entraînement et la validation, par l'empreinte de chaque image ;
- une **configuration invalide refusée** (au moins `epochs=0`, un learning rate hors de $]0, 1[$, une clé inconnue) ;
- le **code de sortie** de `main(argv)`, appelé dans le test (2 pour une configuration invalide, 1 sans données) ;
- un **entraînement fumée** de bout en bout, marqué `@pytest.mark.slow`, sur `synthetic_file` : code 0, le fichier de poids écrit, et le run dans le journal si tu as fait B7.14.

Chaque `assert` a un message dont la **première ligne dit ce qui était attendu** (`f"expected ..., got ..."`), comme dans la fiche : quand l'intégration continue échoue, c'est tout ce que tu lis. Passe `--runs` et `--out` vers `tmp_path` (la fixture de pytest) pour ne rien écrire dans le projet. Si tu reviens après B7.18, ajoute un test de l'API (une réponse 422 pour une image de 28 × 28).

La cellule de vérification écrit tes tests dans `tests/`, les lance par pytest sur ton paquet, fichier par fichier (le nom de chaque test en échec, puis sa raison), et compte tes tests et ceux marqués `slow`. Puis elle les lance sur le **paquet de référence** du chapitre (`solutions/b7_cifarprod`, qui suit le contrat de B7.9 à B7.14 : un test qui y échoue exige autre chose que le contrat) et sur **six versions fautives** de ce paquet, chacune avec une erreur que tes tests doivent voir : le prétraitement qui ne divise plus par 255, `load_npz` qui rend des floats sans vérifier le schéma, une validation qui recouvre l'entraînement, une configuration qui accepte un learning rate hors de $]0, 1[$, `main` qui rend 0 sur une configuration invalide, et une commande `train` qui rend 0 sans écrire ses poids (vue seulement par un test `slow`). Pour chacune, elle dit quel test l'a attrapée, et si la première ligne de son message dit ce qui était attendu.

Dans tes notes : lequel de tes tests attrape le plus de versions fautives, et pourquoi ? Pourquoi un test qui ne fait que « ça ne plante pas » n'en attrape-t-il aucune ? Que verrait l'overfitting d'un mini-batch que les autres ne voient pas (pense à un `zero_grad` oublié) ? Pourquoi l'entraînement fumée est-il marqué `slow`, et quand le lancerais-tu quand même ?

In [ ]:
# Ex B7.15: the tests of your package (written into PROJECT/tests by the check cell)

# tests/conftest.py (given: the fixtures model, data_file, synthetic_file)
CONFTEST_PY = r'''"""Fixtures of the tests of cifarprod: a fresh model, the data file of the project, a small synthetic data file."""

from pathlib import Path

import numpy as np
import pytest
import torch

from cifarprod.data import synthetic
from cifarprod.model import build_model

DATA = Path(__file__).resolve().parents[1] / "data" / "cifar10_2000.npz"


@pytest.fixture
def model():
    torch.manual_seed(0)
    return build_model(widths=(12, 24))


@pytest.fixture
def data_file():
    """The CIFAR-10 subset of the project; the tests that need it are skipped without it (in the CI, for example)."""
    if not DATA.is_file():
        pytest.skip(f"no data file {DATA}")
    return DATA


@pytest.fixture
def synthetic_file(tmp_path):
    """A small .npz file with the schema of the data (96 training and 32 validation random images)."""
    X, y = synthetic(96, seed=0)
    X_val, y_val = synthetic(32, seed=1)
    path = tmp_path / "synthetic.npz"
    np.savez(path, x_train=X, y_train=y, x_val=X_val, y_val=y_val)
    return path
'''

# tests/test_model.py (given: the two tests of the fiche; add yours if you like)
TEST_MODEL_PY = r'''"""Tests of the model: the shape of the output, and a mini-batch learned by heart (the two tests of the fiche, 207.6)."""

import pytest
import torch
import torch.nn.functional as F


@pytest.mark.parametrize("batch", [1, 7])
def test_output_shape(model, batch):
    out = model(torch.zeros(batch, 3, 32, 32))
    assert out.shape == (batch, 10), f"expected ({batch}, 10), got {tuple(out.shape)}"


def test_overfits_one_batch(model):
    g = torch.Generator().manual_seed(0)
    x, y = torch.randn(16, 3, 32, 32, generator=g), torch.arange(16) % 10
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(100):
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
    assert loss.item() < 0.05, f"expected a loss < 0.05 on 16 memorised images, got {loss.item():.3f}"
'''

TEST_DATA_PY = ...   # tests/test_data.py: your tests of the data (the contract)
TEST_CLI_PY = ...    # tests/test_cli.py: your tests of the configuration and of the command line, one marked slow

In [ ]:
ok_15 = {}
PYTEST_ARGS_15 = ["-c", str(PROJECT / "pyproject.toml"), "--rootdir", str(PROJECT), "--strict-markers"]
ENV_15 = {"PYTHONPATH": str(PROJECT / "src")}
TESTS_15 = {"tests/conftest.py": CONFTEST_PY, "tests/test_model.py": TEST_MODEL_PY, "tests/test_data.py": TEST_DATA_PY,
            "tests/test_cli.py": TEST_CLI_PY}
if TEST_DATA_PY is ... or TEST_CLI_PY is ...:
    print("⏳ Ex B7.15 : écris d'abord TEST_DATA_PY et TEST_CLI_PY.")
else:
    with checked("B7.15"):
        wbp.write_files(PROJECT, TESTS_15)
        own_15 = {}
        for rel_15 in ("tests/test_model.py", "tests/test_data.py", "tests/test_cli.py"):
            print(f"— {rel_15}, sur ton paquet :")
            own_15[rel_15] = run_pytest_group(rel_15, root=PROJECT, env=ENV_15, args=PYTEST_ARGS_15)
        ok_15["own"] = all(r.ok for r in own_15.values())
        verdict("B7.15", ok_15["own"], "tes tests passent sur ton paquet.", "tes tests doivent d'abord passer sur ton "
                "paquet (les lignes ci-dessus) ; si c'est ton paquet qui a tort, corrige-le dans sa cellule.")
        collected_15 = {}
        for marker_15 in ("", "slow"):
            listing_15 = wbp.run([sys.executable, "-m", "pytest", "tests/test_data.py", "tests/test_cli.py", "--collect-only",
                                  "-q", "-p", "no:cacheprovider", *PYTEST_ARGS_15, *(["-m", marker_15] if marker_15 else [])],
                                 PROJECT, src=[PROJECT / "src"], env={"COLUMNS": "1000"})
            collected_15[marker_15] = sum("::" in line for line in listing_15.stdout.splitlines())
        ok_15["count"] = collected_15[""] >= 8 and collected_15["slow"] >= 1
        verdict("B7.15", ok_15["count"], f"{collected_15['']} tests dans test_data.py et test_cli.py (paramètres compris), "
                f"dont {collected_15['slow']} marqué(s) slow.", f"{collected_15['']} tests dans test_data.py et test_cli.py, "
                f"{collected_15['slow']} marqué(s) slow : attendu au moins 8 tests (paramètres compris), dont un marqué slow.")
        if not ok_15["own"]:
            raise StopCheck
        base_15 = scratch("mutants")
        reference_15 = wbp.run_tests(wbp.mutant_package(None, base_15 / "reference"), PROJECT / "tests", data=DATA_FILE)
        ok_15["reference"] = reference_15.ok
        verdict("B7.15", reference_15.ok, f"tes tests passent aussi sur le paquet de référence ({reference_15.summary}).",
                f"tes tests doivent passer sur le paquet de référence, qui suit le contrat ({reference_15.summary}) :")
        for name_15, reason_15 in reference_15.failures[:6]:
            print(f"   ❌ {name_15}\n      {reason_15[:300]}")
        if not reference_15.ok:
            print("   un test qui échoue ici exige plus (ou autre chose) que le contrat de B7.9 à B7.14.")
            raise StopCheck
        for k_15, mutant_15 in enumerate(wbp.MUTANTS, start=1):
            package_15 = wbp.mutant_package(mutant_15.key, base_15 / mutant_15.key)
            run_15 = wbp.run_tests(package_15, PROJECT / "tests", data=DATA_FILE,
                                   marker="slow" if mutant_15.slow else "not slow", maxfail=1)
            if run_15.failed + run_15.errors == 0:
                ok_15[mutant_15.key] = False
                print(f"❌ Ex B7.15 : version fautive {k_15} ({mutant_15.what}) : elle passe tous tes tests"
                      + (" marqués slow" if mutant_15.slow else "") + f" ({run_15.summary}). Quel test aurait dû la voir ?")
                continue
            name_15, reason_15 = run_15.failures[0] if run_15.failures else ("?", "")
            ok_15[mutant_15.key] = wbp.says_expected(reason_15)
            verdict("B7.15", ok_15[mutant_15.key], f"version fautive {k_15} ({mutant_15.what}) : attrapée par {name_15} "
                    f"(« {reason_15[:110]} »).", f"version fautive {k_15} ({mutant_15.what}) : attrapée par {name_15}, "
                    f"mais la première ligne de son message ne dit pas ce qui était attendu : « {reason_15[:160]} ».")
if ok_15 and all(ok_15.values()) and len(ok_15) == 3 + len(wbp.MUTANTS):
    print(f"✅ Ex B7.15 : tes tests passent sur un paquet juste et attrapent les {len(wbp.MUTANTS)} versions fautives.")

## Partie B · Le service : les écarts entre entraînement et production, l'artefact, l'API, la latence et l'export

Le modèle sort maintenant du projet pour servir. Tu commences par réparer le service d'un collègue, dont le modèle marche dans son notebook et pas en production (B7.16) : cinq écarts entre la préparation de l'entraînement et celle du service, le training-serving skew du ch. 12. Le modèle du collègue est le VGG 16-32-64 livré par la solution du MP5 (`model.pt` et `model.json`, versionnés avec le dépôt), entraîné sur 45 000 images : la partie B sert un vrai modèle, pas le petit CNN de la partie A.

### Ex B7.16 — Le modèle qui marche dans le notebook et pas en production : cinq écarts à retrouver 🐛 ★★ ⏱️ 30 min
**Objectif :** retrouver, par des diagnostics enchaînés, les écarts entre la préparation de l'entraînement et celle d'un service.  
**Prérequis :** Ex B7.15 · ch. 12 (training-serving skew) · ch. 20 (`eval()`) · fiche §207.7 (« L'artefact du modèle »), §207.10 · **Fil rouge :** CIFAR-10 · **Parcours :** C

Un collègue a entraîné le VGG du MP5 dans son notebook : 85 % sur ces images de test. Il l'a ensuite emballé en service : `save_artifact_16` écrit l'artefact à la fin de l'entraînement ; le service le recharge (`load_service_16`), lit chaque image envoyée par un client (`read_upload_16`, sur un fichier PNG), la prépare (`preprocess_16`) et répond le nom français de la classe (`predict_16`). **Ce que faisait son notebook**, la référence : des images `uint8` RGB `(N, 32, 32, 3)`, divisées par 255, normalisées canal par canal par la moyenne et l'écart-type de **ses images d'entraînement** (`INFO_16["mean"]`, `INFO_16["std"]`), en `float32`, le modèle en mode `eval()`, et la sortie $k$ du modèle est la classe `CLASSES[k]`. En production, le service répond n'importe quoi. **Répare son code sur place** (mêmes noms, mêmes signatures, les docstrings sont le contrat) : cinq écarts s'y cachent, et un écart peut en cacher un autre.

La cellule donnée prépare le modèle du collègue (`MODEL_16`, `INFO_16`, une copie neuve à chaque appel de `wbp.reference_model()`), 200 images de test (`X_16`, `Y_16`) écrites en PNG comme les enverrait un client (`PATHS_16`), ce que le notebook du collègue répond pour elles (`WANT_16`), et les cinq diagnostics. Chaque diagnostic écrit un **nouvel** artefact avec ton `save_artifact_16`, le recharge avec ton `load_service_16`, et ne regarde qu'une partie du service : 1. le service répond-il, et en quel type ; 2. l'image lue d'un fichier ; 3. le prétraitement, sur des images bien lues ; 4. le modèle rechargé, sur des images bien préparées ; 5. les noms rendus pour les 200 images, comparés à ceux du notebook (un **test de non-régression**, ch. 7), puis pour un modèle dont les sorties sont rangées dans un autre ordre. Ils s'enchaînent : le premier qui échoue arrête les suivants (⏳). La cellule suivante montre le service tel qu'il est sur 8 images, puis son accuracy sur les 200.

Dans tes notes : pour chaque écart, le symptôme, la cause, la ligne corrigée, et le test qui l'aurait attrapé avant la mise en production. Pourquoi certains écarts ne faisaient-ils **pas** planter le service ? Lequel en cachait un autre, et pourquoi (pense à ce que fait une normalisation par les statistiques du batch à des pixels de 0 à 255) ? Qu'aurait-il fallu mettre dans l'artefact dès le départ (207.7) ?

In [ ]:
# The colleague's model (the VGG 16-32-64 delivered by the MP5 solution, trained in their notebook on uint8 RGB images
# divided by 255, then normalized with the mean and the std of its training images, in float32; output k = CLASSES[k]),
# 200 test images saved as the PNG files a client uploads, what the evaluation of the notebook gives on them, and the
# five diagnoses, in a row
from safetensors.torch import load_file, save_file

MODEL_16, INFO_16 = wbp.reference_model()               # a new copy at each call: the diagnoses never share it
X_16, Y_16 = wbp.test_images(20)                         # 200 test images (20 per class), uint8 RGB (200, 32, 32, 3)
UPLOADS_16 = scratch("uploads")
PATHS_16 = [wbp.write_png(image, UPLOADS_16 / f"upload_{i:03d}.png") for i, image in enumerate(X_16)]
MEAN_16, STD_16 = np.asarray(INFO_16["mean"], np.float32), np.asarray(INFO_16["std"], np.float32)
FRENCH_16 = {"airplane": "avion", "automobile": "automobile", "bird": "oiseau", "cat": "chat", "deer": "cerf",
             "dog": "chien", "frog": "grenouille", "horse": "cheval", "ship": "bateau", "truck": "camion"}


def offline_input_16(images):
    """The input of the model in the colleague's notebook: uint8 RGB (N, 32, 32, 3) -> float32 (N, 3, 32, 32)."""
    x = (np.asarray(images, np.float32) / np.float32(255) - MEAN_16) / STD_16
    return torch.from_numpy(np.ascontiguousarray(x.transpose(0, 3, 1, 2)))


with torch.inference_mode():
    OFFLINE_16 = MODEL_16(offline_input_16(X_16)).argmax(dim=1).numpy()      # the evaluation of the notebook
WANT_16 = [FRENCH_16[CLASSES[k]] for k in OFFLINE_16]                         # what the service should answer
print(f"Dans le notebook du collègue : accuracy {fr((OFFLINE_16 == Y_16).mean(), 3)} sur ces 200 images de test.")


def fresh_service_16(model=None, classes=CLASSES):
    """(model, meta) of the service, from a NEW artifact written by YOUR save_artifact_16 (the colleague's model, or
    `model` whose outputs are the classes `classes`), loaded by YOUR load_service_16."""
    folder = scratch("artifact")
    save_artifact_16(model if model is not None else wbp.reference_model()[0], folder, list(INFO_16["mean"]),
                     list(INFO_16["std"]), list(classes))
    return load_service_16(folder)


def diagnose_16(ex_id="B7.16"):
    """The five diagnoses of the service, in a row: the first one that fails stops the chain (one gap can hide
    another). Returns True when the five pass."""

    def point1():                                         # does the service answer?
        model, meta = fresh_service_16()
        seen = []
        handle = model.register_forward_pre_hook(lambda module, args: seen.append(args[0].dtype))
        try:
            names = predict_16(model, meta, PATHS_16[:8])
        except Exception as error:  # noqa: BLE001 - the learner's code: the symptom
            return False, f"le service plante sur 8 images envoyées : {type(error).__name__} : {str(error)[:160]}", \
                "lis la dernière ligne de l'erreur : quel tenseur ne va pas avec les poids du modèle ?"
        finally:
            handle.remove()
        if not (isinstance(names, list) and len(names) == 8 and all(isinstance(n, str) for n in names)):
            return False, f"predict_16 rend {type(names).__name__} {str(names)[:80]}", "attendu une liste de 8 noms (str)"
        weights = {p.dtype for p in model.parameters()}
        if weights != {torch.float32} or set(seen) != {torch.float32}:
            return False, (f"le modèle du service a des poids {sorted(map(str, weights))} et reçoit {[str(d) for d in seen]}, "
                           "alors que l'artefact et l'entraînement sont en float32"), "où ce type apparaît-il ?"
        return True, "le service répond 8 noms, en float32 comme l'entraînement", ""

    def point2():                                         # the image read from the file
        image = read_upload_16(PATHS_16[0])
        got = np.asarray(image)
        want = X_16[0].astype(np.float32) / 255
        if got.shape != (32, 32, 3):
            return False, f"l'image lue a la forme {got.shape}", "attendu (32, 32, 3), comme à l'entraînement"
        got = got.astype(np.float64)
        if np.allclose(got, want, atol=1e-6):
            return True, "l'image lue est celle de l'entraînement : RGB, des floats dans [0, 1]", ""
        symptoms, scaled = [], got
        if got.max() > 1.5:
            symptoms.append(f"ses valeurs vont de {got.min():.0f} à {got.max():.0f}, alors que l'entraînement voit des "
                            "pixels dans [0, 1]")
            scaled = got / 255
        if not np.allclose(scaled, want, atol=1e-6) and np.allclose(scaled, want[..., ::-1], atol=1e-6):
            symptoms.append("son canal 0 est le canal 2 de l'image envoyée (le bleu à la place du rouge)")
        elif not np.allclose(scaled, want, atol=1e-6):
            symptoms.append(f"elle s'écarte de l'image envoyée de {np.abs(scaled - want).max():.3f}")
        return False, "l'image lue n'est pas celle que voit l'entraînement : " + " ; ".join(symptoms), \
            "compare-la à X_16[0] / 255 : que rend la fonction de lecture ?"

    def point3():                                         # the preprocessing
        model, meta = fresh_service_16()
        images = X_16[:8].astype(np.float32) / 255
        batch = torch.as_tensor(preprocess_16(images.copy(), meta)).double()
        if tuple(batch.shape) != (8, 3, 32, 32):
            return False, f"preprocess_16 de 8 images rend la forme {tuple(batch.shape)}", "attendu (8, 3, 32, 32)"
        alone = torch.as_tensor(preprocess_16(images[:1].copy(), meta)).double()
        want = offline_input_16(X_16[:8]).double()
        if not torch.allclose(alone[0], batch[0], atol=1e-5):
            return False, (f"une image prétraitée seule n'a pas les mêmes valeurs que dans un batch de 8 (écart "
                           f"{float((alone[0] - batch[0]).abs().max()):.3f}) : son prétraitement dépend des autres images "
                           "envoyées"), "d'où viennent la moyenne et l'écart-type qu'utilise le service ?"
        if not torch.allclose(batch, want, atol=1e-4):
            return False, (f"le prétraitement du service s'écarte de celui de l'entraînement de "
                           f"{float((batch - want).abs().max()):.3f}"), "relis comment le notebook préparait les images"
        return True, "le prétraitement est celui de l'entraînement, image par image", ""

    def point4():                                         # the model of the service
        model, meta = fresh_service_16()
        x = offline_input_16(X_16[:8])
        before = {k: v.clone() for k, v in model.state_dict().items()}
        with torch.inference_mode():
            together, alone = model(x), model(x[:1])
        changed = any(not torch.equal(before[k], v) for k, v in model.state_dict().items())
        if changed or not torch.allclose(together[:1], alone, atol=1e-5):
            return False, ("la même image, bien prétraitée, reçoit d'autres scores seule et dans un batch de 8 (écart "
                           f"{float((together[:1] - alone).abs().max()):.3f})"
                           + (", et chaque appel change les statistiques internes du modèle (running_mean)" if changed else "")), \
                "compare le modèle que rend load_service_16 à celui de l'évaluation hors ligne : qu'est-ce qui les distingue ?"
        with torch.inference_mode():
            want = MODEL_16(x)
        if not torch.allclose(together, want, atol=1e-4):
            return False, f"les scores du service s'écartent de ceux du notebook de {float((together - want).abs().max()):.3f}", \
                "les poids chargés sont-ils ceux de l'artefact ?"
        return True, "le modèle du service donne les scores du notebook, seule ou en batch", ""

    def point5():                                         # the names returned: the non-regression test
        model, meta = fresh_service_16()
        names = predict_16(model, meta, PATHS_16)
        agree = sum(a == b for a, b in zip(names, WANT_16))
        if agree != len(WANT_16):
            pairs = {}
            for got, want in zip(names, WANT_16):
                if got != want:
                    pairs[(want, got)] = pairs.get((want, got), 0) + 1
            (want, got), count = max(pairs.items(), key=lambda item: item[1])
            return False, (f"sur 200 images, le service ne rend le nom du notebook que pour {agree} ; par exemple, "
                           f"{count} images que le notebook classe « {want} » reçoivent « {got} »"), \
                "le nom que rend le service correspond-il à la sortie du modèle ?"
        perm = np.random.default_rng(16).permutation(10)
        shuffled = wbp.reference_model()[0]
        with torch.no_grad():
            shuffled[-1].weight.copy_(shuffled[-1].weight[perm].clone())
            shuffled[-1].bias.copy_(shuffled[-1].bias[perm].clone())
        model, meta = fresh_service_16(shuffled, [CLASSES[k] for k in perm])
        agree = sum(a == b for a, b in zip(predict_16(model, meta, PATHS_16), WANT_16))
        if agree != len(WANT_16):
            return False, (f"le même modèle, ses sorties rangées dans un autre ordre (et classes dans cet ordre), ne "
                           f"reçoit le bon nom que pour {agree} images sur 200"), \
                "d'où le service tient-il l'ordre des classes ?"
        return True, ("les 200 noms sont ceux du notebook, même pour un modèle dont les sorties sont rangées dans un "
                      "autre ordre"), ""

    points = [("le service répond", point1), ("l'image lue", point2), ("le prétraitement", point3),
              ("le modèle du service", point4), ("les noms rendus (test de non-régression)", point5)]
    for k, (name, run) in enumerate(points, start=1):
        try:
            with wbp.torch_state_kept():
                ok, measured, why = run()
        except NotImplementedError:
            raise
        except Exception as error:  # noqa: BLE001 - the learner's code: one ❌ line, then the chain stops
            ok, measured, why = False, f"une erreur {type(error).__name__} : {str(error)[:200]}", "relis la fonction en cause"
        if ok:
            print(f"✅ Ex {ex_id} : point {k}, {name} ({measured}).")
        else:
            print(f"❌ Ex {ex_id} : point {k}, {name} : {measured} ; {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex {ex_id} : point {later} : corrige d'abord le point {k}.")
            return False
    print(f"✅ Ex {ex_id} : les cinq diagnostics passent : le service répond comme le notebook.")
    return True

In [ ]:
# The colleague's service, to repair IN PLACE (same names, same signatures). It works in their notebook, not in production.
def save_artifact_16(model, folder, mean, std, classes):
    """Training side: the artifact of `model` in `folder`: model.safetensors (the weights) and meta.json (what the
    service needs). mean, std: the normalization learned on the training images; classes: the names of the outputs,
    in their order."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    save_file(model.state_dict(), str(folder / "model.safetensors"))
    meta = {"arch": "vgg", "widths": [16, 32, 64], "mean": list(mean), "std": list(std)}
    (folder / "meta.json").write_text(json.dumps(meta), encoding="utf-8")


def load_service_16(folder):
    """Service side: (model, meta), the model rebuilt from the artifact, ready to predict."""
    meta = json.loads((Path(folder) / "meta.json").read_text(encoding="utf-8"))
    model = wbp.build_reference(meta["arch"], meta["widths"])
    model.load_state_dict(load_file(str(Path(folder) / "model.safetensors")))
    return model, meta


def read_upload_16(path):
    """Service side: an uploaded image file, as the array the model was trained on."""
    return wbp.imread_bgr(path)


def preprocess_16(images, meta):
    """Service side: the images (N, 32, 32, 3) read by read_upload_16, as the input tensor (N, 3, 32, 32) of the model."""
    x = np.asarray(images, dtype=np.float64)                      # double precision, to be safe
    mean, std = x.mean(axis=(0, 1, 2)), x.std(axis=(0, 1, 2))
    x = (x - mean) / std
    return torch.from_numpy(x).permute(0, 3, 1, 2)


def predict_16(model, meta, paths):
    """Service side: the French name of the class of each uploaded image (a list of str)."""
    images = np.stack([read_upload_16(p) for p in paths])
    with torch.inference_mode():
        k = model(preprocess_16(images, meta)).argmax(dim=1)
    names = sorted(FRENCH_16.values())
    return [names[i] for i in k.tolist()]

In [ ]:
repaired_16_ok = False
with checked("B7.16"):
    repaired_16_ok = diagnose_16()

In [ ]:
# The service, as it is now (it shows the symptoms before the repair), on 8 uploads and on the 200 images
try:
    with wbp.torch_state_kept():
        model_16, meta_16 = fresh_service_16()
        names_16 = predict_16(model_16, meta_16, PATHS_16)
except Exception as error_16:  # noqa: BLE001 - the learner's code
    names_16 = None
    print(f"⏳ Ex B7.16 : le service lève {type(error_16).__name__} ({str(error_16)[:160]}) : rien à montrer.")
if isinstance(names_16, list) and len(names_16) == len(PATHS_16):
    print("image   vraie classe   notebook   service")
    for i_16 in range(8):
        print(f"{i_16:5d}   {FRENCH_16[CLASSES[Y_16[i_16]]]:12s}   {WANT_16[i_16]:10s} {names_16[i_16]}")
    truth_16 = [FRENCH_16[CLASSES[k]] for k in Y_16]
    print(f"accuracy sur 200 images : notebook {fr((OFFLINE_16 == Y_16).mean(), 3)}, service "
          f"{fr(np.mean([a == b for a, b in zip(names_16, truth_16)]), 3)} ; accord service-notebook "
          f"{sum(a == b for a, b in zip(names_16, WANT_16))} sur 200.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire où va chaque fichier d'un projet de ML (code, tests, configuration, données) et pourquoi un notebook ne suffit pas ?
2. Sais-tu rapporter une accuracy sur cinq graines avec son intervalle, et comparer deux modèles graine par graine ?
3. Sais-tu calculer un p50, un p95 et un débit, et dimensionner un service à la louche ?
4. Sais-tu calculer un PSI, dire ce qu'il mesure, et ce que tu ferais d'une alerte ?

**Pour aller plus loin** : l'article de Sculley et coll. (📄 B7.8) et le ML Test Score de Breck et coll., cités dans la fiche. La suite : le bonus B8, l'apprentissage par renforcement moderne.

*Les exercices B7.17 à B7.21 et B7.22 à B7.25 seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B7` pour obtenir le notebook complet.*